# TOFU 3B NPO-LoRA pilot

This private notebook starts with a bounded, one-step smoke on eight pinned TOFU forget examples and sampled retain examples. It loads the verified Llama 3.2 3B full checkpoint on Kaggle T4 x2, performs one LoRA optimizer update, checks that the adapter actually changes, then discards that smoke adapter. The default notebook does **not** start the 10-epoch pilot.

Only after the smoke receipt passes should a new notebook version be built with `--run-full-training` and pushed. The pilot is a single-seed LoRA adaptation, not full-parameter NPO or confirmatory evidence. It does not train on holdout. All model and dataset revisions, hashes, thresholds, and stopping rules are locked in the included protocol. No Hugging Face token is needed for these public Open-Unlearning checkpoints.

In [ ]:
import importlib.metadata as metadata
import json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_FULL_TRAINING = True
print({'full_training_enabled': RUN_FULL_TRAINING, 'notebook_mode': 'full pilot' if RUN_FULL_TRAINING else 'bounded smoke only'})


In [ ]:
requirements = Path('/kaggle/working/requirements-npo-pilot.txt')
requirements.write_text('# Kaggle runtime packages used by the bounded TOFU NPO-LoRA pilot.\n# Keep the preinstalled CUDA-enabled torch build supplied by Kaggle.\ntransformers==5.22.1\naccelerate==1.15.0\npeft==0.21.2\nscipy==1.16.3\nrouge-score==0.1.2\n', encoding='utf-8')
expected = {'transformers': '5.22.1', 'accelerate': '1.15.0', 'peft': '0.21.2', 'scipy': '1.16.3', 'rouge-score': '0.1.2'}
mismatch = []
for package, version in expected.items():
    try: actual = metadata.version(package)
    except metadata.PackageNotFoundError: actual = None
    if actual != version: mismatch.append(package + '==' + version)
if mismatch:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements)], check=True)
versions = {package: metadata.version(package) for package in expected}
if versions != expected: raise RuntimeError(f'Pinned package check failed: {versions}')
torchao_version = None
try: torchao_version = metadata.version('torchao')
except metadata.PackageNotFoundError: pass
if torchao_version is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchao'], check=True, capture_output=True, text=True)
    try: metadata.version('torchao'); raise RuntimeError('torchao remained installed after removal')
    except metadata.PackageNotFoundError: pass
import importlib.util
if importlib.util.find_spec('torchao') is not None: raise RuntimeError('torchao module remains importable after removal')
import torch
from transformers import LlamaConfig, LlamaForCausalLM
from peft import LoraConfig, get_peft_model
tiny_config = LlamaConfig(vocab_size=32, hidden_size=16, intermediate_size=32, num_hidden_layers=1, num_attention_heads=2, num_key_value_heads=2)
tiny = LlamaForCausalLM(tiny_config)
tiny = get_peft_model(tiny, LoraConfig(r=2, lora_alpha=4, lora_dropout=0.0, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], bias='none', task_type='CAUSAL_LM'))
tiny_ids = torch.tensor([[1,2,3,4]])
tiny_loss = tiny(input_ids=tiny_ids, labels=tiny_ids, use_cache=False).loss
tiny_loss.backward()
tiny_trainable = [parameter for parameter in tiny.parameters() if parameter.requires_grad]
tiny_grad = sum(float(parameter.grad.float().abs().sum()) for parameter in tiny_trainable if parameter.grad is not None)
if not torch.isfinite(tiny_loss) or not tiny_trainable or not torch.isfinite(torch.tensor(tiny_grad)) or tiny_grad <= 0: raise RuntimeError('PEFT LoRA dispatch preflight failed')
print({'packages': versions, 'torch': metadata.version('torch'), 'torchao': 'absent', 'peft_lora_dispatch_smoke': 'PASS', 'tiny_loss': float(tiny_loss.detach()), 'tiny_adapter_gradient_l1': tiny_grad})


In [ ]:
import torch
import urllib.request
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2. Refusing model loading before two-GPU validation.')
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
if gpu_names.count('Tesla T4') != 2:
    raise RuntimeError(f'Protocol requires exactly two Tesla T4 GPUs; found {gpu_names}')
with urllib.request.urlopen('https://huggingface.co', timeout=20) as response: internet_status = response.status
if internet_status != 200: raise RuntimeError(f'Internet preflight returned HTTP {internet_status}')
print({'gpu_names': gpu_names, 'total_memory_bytes': [torch.cuda.get_device_properties(i).total_memory for i in range(2)], 'internet_http_status': internet_status})


In [ ]:
upstream = Path('/kaggle/working/open-unlearning')
commit = '2d98027677f3468d52b6baa3fc4f2b9f2dc08820'
if not upstream.exists(): subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/locuslab/open-unlearning.git', str(upstream)], check=True)
subprocess.run(['git', '-C', str(upstream), 'fetch', '--depth', '1', 'origin', commit], check=True)
subprocess.run(['git', '-C', str(upstream), 'checkout', '--detach', commit], check=True)
actual = subprocess.check_output(['git', '-C', str(upstream), 'rev-parse', 'HEAD'], text=True).strip()
if actual != commit: raise RuntimeError(f'Wrong upstream commit: {actual}')
print({'open_unlearning_commit': actual})


In [ ]:
%%writefile /kaggle/working/tofu_official_eval.py
#!/usr/bin/env python3
"""Evaluate pinned TOFU checkpoints with the core Open-Unlearning metrics.

This is inference-only. Full checkpoints receive answer likelihood, greedy
generation/ROUGE-L, and truth-ratio evaluation on the canonical TOFU sets.
The paired retain90 oracle checkpoints are scored on forget truth ratios only
to compute the benchmark's forget-quality comparison.
"""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import os
import random
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from scipy.stats import ks_2samp
from transformers import AutoModelForCausalLM, AutoTokenizer


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
UPSTREAM_REVISION = "2d98027677f3468d52b6baa3fc4f2b9f2dc08820"
PROFILE_QA_COUNT = 20
BOOTSTRAP_SEED = 42
BOOTSTRAP_DRAWS = 5000
MAX_LENGTH = 512
SYSTEM_PROMPT = "You are a helpful assistant."
TEMPLATE_DATE = "10 Apr 2025"
MAX_NEW_TOKENS = 200
IGNORE_INDEX = -100
EPSILON = 1e-10

CHECKPOINTS = {
    "llama_3_2_3b_full": {
        "label": "Llama 3.2 3B full",
        "family": "llama_3_2_3b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    },
    "llama_3_2_3b_retain90": {
        "label": "Llama 3.2 3B retain90 oracle",
        "family": "llama_3_2_3b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
        "revision": "2c35819e9a731d096df9abd6efbcc361870c5333",
    },
    "llama_3_1_8b_full": {
        "label": "Llama 3.1 8B full",
        "family": "llama_3_1_8b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "revision": "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    },
    "llama_3_1_8b_retain90": {
        "label": "Llama 3.1 8B retain90 oracle",
        "family": "llama_3_1_8b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90",
        "revision": "f02956a9f25a18f38ad7808fbefb2f957347273e",
    },
}

DATA_FILES = (
    "forget10.json",
    "forget10_perturbed.json",
    "retain90.json",
    "retain_perturbed.json",
    "holdout10.json",
    "real_authors_perturbed.json",
    "world_facts_perturbed.json",
)


def get_bytes(url: str) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-official-eval/1.0"})
    with urlopen(request, timeout=60) as response:
        return response.read()


def load_data() -> tuple[dict[str, list[dict]], dict[str, str]]:
    rows_by_file = {}
    hashes = {}
    for filename in DATA_FILES:
        url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
        raw = get_bytes(url)
        rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
        if any(not isinstance(row, dict) for row in rows):
            raise ValueError(f"{filename} contains a non-object JSONL row")
        if any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
            raise ValueError(f"{filename} has a missing question/answer string")
        rows_by_file[filename] = rows
        hashes[filename] = hashlib.sha256(raw).hexdigest()
    return rows_by_file, hashes


def build_eval_sets(data: dict[str, list[dict]]) -> dict[str, list[dict]]:
    """Build canonical evaluation rows and attach source profile blocks."""
    retain_source = data["retain90.json"]
    retain_index = collections.defaultdict(list)
    retain_question_answers = collections.defaultdict(set)
    for i, row in enumerate(retain_source):
        retain_index[(row["question"], row["answer"])].append(i)
        retain_question_answers[row["question"]].add(row["answer"])
    collision_indices = {
        i
        for i, row in enumerate(retain_source)
        if len(retain_question_answers[row["question"]]) > 1
    }

    def basic(rows: list[dict], split: str, source_indices: list[int] | None = None) -> list[dict]:
        result = []
        for i, row in enumerate(rows):
            source_i = i if source_indices is None else source_indices[i]
            result.append({
                "split": split,
                "eval_row_index": i,
                "source_row_index": source_i,
                "profile_block": source_i // PROFILE_QA_COUNT if split in {"forget10", "retain90", "holdout10"} else None,
                "question": row["question"],
                "answer": row["answer"],
                "ambiguous_question_collision": split == "retain90" and source_i in collision_indices,
                "paraphrased_answer": row.get("paraphrased_answer"),
                "perturbed_answers": row.get("perturbed_answer"),
            })
        return result

    forget = data["forget10_perturbed.json"]
    forget_source = data["forget10.json"]
    if len(forget) != len(forget_source) or any(
        (a["question"], a["answer"]) != (b["question"], b["answer"])
        for a, b in zip(forget_source, forget)
    ):
        raise ValueError("forget10_perturbed no longer matches forget10 in source order")

    retain = data["retain_perturbed.json"]
    retain_source_indices = []
    for row in retain:
        matches = retain_index.get((row["question"], row["answer"]), [])
        if len(matches) != 1:
            raise ValueError("A canonical retain_perturbed row does not map uniquely to retain90")
        retain_source_indices.append(matches[0])
    if sorted(retain_source_indices) != list(range(20 * PROFILE_QA_COUNT)):
        raise ValueError("Canonical retain_perturbed is not the pinned first-20-profile retain sample")

    return {
        "forget10": basic(forget, "forget10"),
        "retain90": basic(retain, "retain90", retain_source_indices),
        "holdout10": basic(data["holdout10.json"], "holdout10"),
        "real_authors": basic(data["real_authors_perturbed.json"], "real_authors"),
        "world_facts": basic(data["world_facts_perturbed.json"], "world_facts"),
    }


def ids(value) -> list[int]:
    if hasattr(value, "keys") and "input_ids" in value:
        value = value["input_ids"]
    if isinstance(value, torch.Tensor):
        value = value.tolist()
    if value and isinstance(value[0], list):
        value = value[0]
    return [int(x) for x in value]


def encode_answer(tokenizer, question: str, answer: str) -> tuple[list[int], list[int]]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    kwargs = {"date_string": TEMPLATE_DATE}
    full_ids = ids(tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=False, **kwargs
    ))
    prompt_ids = ids(tokenizer.apply_chat_template(
        conversation[:-1], tokenize=True, add_generation_prompt=True, **kwargs
    ))
    if not full_ids or len(prompt_ids) > len(full_ids):
        raise ValueError("Invalid chat-template tokenization")
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokenization is not an exact prefix of the full conversation")
    if tokenizer.eos_token_id is not None and full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(int(tokenizer.eos_token_id))
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    if len(labels) != len(full_ids) or not any(x != IGNORE_INDEX for x in labels[1:]):
        raise ValueError("Assistant answer mask is empty or misaligned")
    return full_ids, labels


def encode_prompt(tokenizer, question: str) -> list[int]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    result = ids(tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        date_string=TEMPLATE_DATE,
    ))
    if not result:
        raise ValueError("Chat-template prompt was empty")
    return result


def collate_scoring(rows: list[tuple[list[int], list[int]]], pad_id: int):
    width = max(len(x[0]) for x in rows)
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    labels = torch.full((len(rows), width), IGNORE_INDEX, dtype=torch.long)
    for i, (tokens, targets) in enumerate(rows):
        n = len(tokens)
        input_ids[i, :n] = torch.tensor(tokens)
        attention[i, :n] = 1
        labels[i, :n] = torch.tensor(targets)
    return input_ids, attention, labels


def score_answers(model, encoded: list[tuple[list[int], list[int]]], pad_id: int, batch_size: int) -> list[dict]:
    device = model.get_input_embeddings().weight.device
    output_scores = []
    offset = 0
    batches = 0
    while offset < len(encoded):
        size = min(batch_size, len(encoded) - offset)
        batch = encoded[offset:offset + size]
        input_ids, attention, labels = collate_scoring(batch, pad_id)
        try:
            with torch.inference_mode():
                out = model(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    use_cache=False,
                )
                logits = out.logits[:, :-1, :].float()
                targets = labels[:, 1:].to(logits.device)
                losses = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="none",
                ).reshape(targets.shape)
                tokens_per_row = (targets != IGNORE_INDEX).sum(dim=1)
                if torch.any(tokens_per_row == 0):
                    raise ValueError("Evaluation example has no scored assistant tokens")
                mean_loss = losses.sum(dim=1) / tokens_per_row
                for i in range(size):
                    value = float(mean_loss[i].item())
                    output_scores.append({
                        "avg_answer_token_loss": value,
                        "answer_probability": math.exp(-value),
                        "scored_answer_tokens": int(tokens_per_row[i].item()),
                    })
            offset += size
            batches += 1
            if batches % 100 == 0 or offset == len(encoded):
                print(f"Scored {offset}/{len(encoded)} answer candidates", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during scoring; reducing batch size to {batch_size}", flush=True)
    return output_scores


def collate_prompts(rows: list[list[int]], pad_id: int):
    width = max(map(len, rows))
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    for i, row in enumerate(rows):
        input_ids[i, -len(row):] = torch.tensor(row)
        attention[i, -len(row):] = 1
    return input_ids, attention


def generate_answers(model, tokenizer, prompts: list[list[int]], batch_size: int) -> list[str]:
    device = model.get_input_embeddings().weight.device
    generations = []
    offset = 0
    batches = 0
    model.config.use_cache = True
    while offset < len(prompts):
        size = min(batch_size, len(prompts) - offset)
        current = prompts[offset:offset + size]
        input_ids, attention = collate_prompts(current, int(tokenizer.pad_token_id))
        try:
            with torch.inference_mode():
                output = model.generate(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    use_cache=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = output[:, input_ids.shape[1]:]
            texts = tokenizer.batch_decode(
                new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            generations.extend(text.strip() for text in texts)
            offset += size
            batches += 1
            if batches % 25 == 0 or offset == len(prompts):
                print(f"Generated {offset}/{len(prompts)} answers", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during generation; reducing batch size to {batch_size}", flush=True)
    return generations


def build_jobs(eval_sets: dict[str, list[dict]], role: str):
    score_jobs = []
    generation_jobs = []
    for split, rows in eval_sets.items():
        for row in rows:
            key = (split, row["eval_row_index"])
            if role == "full" or split == "forget10":
                score_jobs.append({**row, "variant": "answer", "candidate_answer": row["answer"]})
                if row["paraphrased_answer"]:
                    score_jobs.append({**row, "variant": "paraphrased_answer", "candidate_answer": row["paraphrased_answer"]})
                for i, candidate in enumerate(row["perturbed_answers"] or []):
                    score_jobs.append({**row, "variant": f"perturbed_{i}", "candidate_answer": candidate})
            if role == "full":
                generation_jobs.append(row)
    return score_jobs, generation_jobs


def score_rows_by_key(jobs: list[dict], scores: list[dict]) -> dict[tuple[str, int], dict[str, dict]]:
    result = collections.defaultdict(dict)
    for job, score in zip(jobs, scores):
        key = (job["split"], job["eval_row_index"])
        if job["variant"] in result[key]:
            raise ValueError(f"Duplicate answer candidate score for {key}/{job['variant']}")
        result[key][job["variant"]] = score
    return result


def bootstrap_cluster_ci(rows: list[dict], field: str, seed: int = BOOTSTRAP_SEED) -> list[float] | None:
    groups = collections.defaultdict(list)
    for row in rows:
        if row.get("profile_block") is not None and row.get(field) is not None:
            groups[int(row["profile_block"])].append(float(row[field]))
    if not groups:
        return None
    group_values = list(groups.values())
    rng = random.Random(seed)
    estimates = []
    for _ in range(BOOTSTRAP_DRAWS):
        sampled = [rng.choice(group_values) for _ in group_values]
        values = [value for group in sampled for value in group]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return [estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))], estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))]]


def mean_field(rows: list[dict], field: str) -> float | None:
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None


def summarize_examples(examples: list[dict]) -> dict:
    summary = {"n_questions": len(examples), "n_profile_blocks": len({r["profile_block"] for r in examples if r.get("profile_block") is not None})}
    fields = (
        "answer_probability",
        "rougeL_recall",
        "truth_ratio",
        "truth_ratio_score",
        "normalized_option_probability",
    )
    for field in fields:
        value = mean_field(examples, field)
        if value is not None:
            summary[f"mean_{field}"] = value
            ci = bootstrap_cluster_ci(examples, field)
            if ci is not None:
                summary[f"profile_cluster_bootstrap_95pct_ci_{field}"] = ci
    return summary


def make_metrics(
    rows: list[dict],
    split: str,
    score_map: dict[tuple[str, int], dict[str, dict]],
    generations: dict[tuple[str, int], str],
    tokenizer,
    rouge: rouge_scorer.RougeScorer,
) -> list[dict]:
    metrics = []
    for row in rows:
        key = (split, row["eval_row_index"])
        scores = score_map.get(key, {})
        if "answer" not in scores:
            raise ValueError(f"Missing gold-answer score for {key}")
        output = {
            "split": split,
            "eval_row_index": row["eval_row_index"],
            "source_row_index": row["source_row_index"],
            "profile_block": row["profile_block"],
            "question_sha256": hashlib.sha256(row["question"].encode("utf-8")).hexdigest(),
            "source_answer": row["answer"],
            "reference_answer": row["answer"],
            "ambiguous_question_collision": row["ambiguous_question_collision"],
            "answer_probability": scores["answer"]["answer_probability"],
            "answer_token_loss": scores["answer"]["avg_answer_token_loss"],
            "answer_scored_tokens": scores["answer"]["scored_answer_tokens"],
        }
        if "paraphrased_answer" in scores:
            para = scores["paraphrased_answer"]
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"] or []))]
            output["paraphrased_answer_probability"] = para["answer_probability"]
            output["paraphrased_answer_token_loss"] = para["avg_answer_token_loss"]
            output["perturbed_answer_probabilities"] = [x["answer_probability"] for x in wrong]
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            if wrong:
                # Match the pinned Open-Unlearning implementation: exponentiate
                # the mean per-candidate answer loss, i.e. the geometric mean of
                # perturbed-answer probabilities.
                geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
                ratio = geometric_wrong_probability / (para["answer_probability"] + EPSILON)
                output["truth_ratio"] = ratio
                output["truth_ratio_score"] = min(ratio, 1.0 / (ratio + EPSILON)) if split == "forget10" else max(0.0, 1.0 - ratio)
        elif row["perturbed_answers"]:
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"]))]
            wrong_probabilities = [x["answer_probability"] for x in wrong]
            geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
            ratio = geometric_wrong_probability / (scores["answer"]["answer_probability"] + EPSILON)
            output["perturbed_answer_probabilities"] = wrong_probabilities
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            output["truth_ratio"] = ratio
            output["truth_ratio_score"] = max(0.0, 1.0 - ratio)
            output["normalized_option_probability"] = scores["answer"]["answer_probability"] / (
                scores["answer"]["answer_probability"] + sum(wrong_probabilities) + EPSILON
            )
        if key in generations:
            text = generations[key]
            full_ids, _ = encode_answer(tokenizer, row["question"], row["answer"])
            prompt_ids = encode_prompt(tokenizer, row["question"])
            input_text = tokenizer.decode(
                prompt_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            full_text = tokenizer.decode(
                full_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            reference = full_text.replace(input_text, "").strip()
            output["generation"] = text
            output["reference_answer"] = reference
            output["rougeL_recall"] = rouge.score(reference, text)["rougeL"].recall
        metrics.append(output)
    return metrics


def model_utility(metrics_by_split: dict[str, list[dict]]) -> dict:
    components = {
        "retain_probability": mean_field(metrics_by_split["retain90"], "answer_probability"),
        "retain_rougeL_recall": mean_field(metrics_by_split["retain90"], "rougeL_recall"),
        "retain_truth_ratio_score": mean_field(metrics_by_split["retain90"], "truth_ratio_score"),
        "real_authors_normalized_option_probability": mean_field(metrics_by_split["real_authors"], "normalized_option_probability"),
        "real_authors_rougeL_recall": mean_field(metrics_by_split["real_authors"], "rougeL_recall"),
        "real_authors_truth_ratio_score": mean_field(metrics_by_split["real_authors"], "truth_ratio_score"),
        "world_facts_normalized_option_probability": mean_field(metrics_by_split["world_facts"], "normalized_option_probability"),
        "world_facts_rougeL_recall": mean_field(metrics_by_split["world_facts"], "rougeL_recall"),
        "world_facts_truth_ratio_score": mean_field(metrics_by_split["world_facts"], "truth_ratio_score"),
    }
    if any(value is None for value in components.values()):
        raise ValueError(f"Cannot compute complete TOFU model utility; missing component: {components}")
    if any(value <= 0 for value in components.values()):
        aggregate = 0.0
    else:
        aggregate = len(components) / sum(1.0 / value for value in components.values())
    return {"components": components, "harmonic_mean": aggregate}


def write_reports(output_dir: Path) -> None:
    keys = ("llama_3_2_3b_full", "llama_3_1_8b_full")
    results = {key: json.loads((output_dir / f"{key}_official_eval.json").read_text(encoding="utf-8")) for key in keys}
    summaries = {key: results[key]["summary"] for key in keys}
    split_labels = {
        "forget10": "Forget10",
        "retain90": "Canonical retain eval",
        "holdout10": "Holdout10",
        "real_authors": "Real authors",
        "world_facts": "World facts",
    }
    lines = [
        "# TOFU official-metric pre-unlearning baseline",
        "",
        "**Status:** complete and inference-only. It evaluates the published full checkpoints and paired retain90 oracle checkpoints. No NPO training or research adapter was run.",
        "",
        "The metric formulas, chat preprocessing, and generated-reference extraction follow the pinned Open-Unlearning implementation. `answer_probability` is mean per-answer-token probability; `rougeL_recall` scores greedy generated answers; Truth Ratio compares the geometric mean likelihood of perturbed answers with the paraphrased correct answer (or source correct answer for real-author/world-fact utility). Kaggle T4 uses FP16 + SDPA here, while upstream configs specify BF16 + FlashAttention 2; formulas and prompts match, but numerical generations are not bitwise-identical to the upstream runtime.",
        "",
        "## Full-checkpoint metrics",
        "",
        "| Split | Model | Questions / profile blocks | Mean answer probability (95% profile CI) | Mean ROUGE-L recall (95% profile CI) | Mean truth ratio | TOFU truth-ratio score | Normalized option probability |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for split, label in split_labels.items():
        for key in keys:
            summary = summaries[key][split]
            def render(metric: str, ci_key: str | None = None) -> str:
                val = summary.get(f"mean_{metric}")
                if val is None:
                    return "—"
                if ci_key and ci_key in summary:
                    low, high = summary[ci_key]
                    return f"{val:.4f} [{low:.4f}, {high:.4f}]"
                return f"{val:.4f}"
            n = f"{summary['n_questions']} / {summary['n_profile_blocks']}" if summary["n_profile_blocks"] else f"{summary['n_questions']} / —"
            lines.append(
                f"| {label} | {results[key]['model']['label']} | {n} | "
                f"{render('answer_probability', 'profile_cluster_bootstrap_95pct_ci_answer_probability')} | "
                f"{render('rougeL_recall', 'profile_cluster_bootstrap_95pct_ci_rougeL_recall')} | "
                f"{render('truth_ratio')} | {render('truth_ratio_score')} | {render('normalized_option_probability')} |"
            )
    lines += ["", "## Composite utility and forget-quality comparison", ""]
    lines.append("| Model | TOFU model-utility harmonic mean | Forget-quality KS p-value vs retain90 oracle |")
    lines.append("|---|---:|---:|")
    for key in keys:
        utility = results[key]["model_utility"]["harmonic_mean"]
        fq = results[key].get("forget_quality_vs_retain90_oracle", {}).get("p_value")
        lines.append(f"| {results[key]['model']['label']} | {utility:.6f} | {fq:.6g}" if fq is not None else f"| {results[key]['model']['label']} | {utility:.6f} | pending oracle result |")

    lines += [
        "",
        "## Retain-question collision sensitivity",
        "",
        "The pinned `retain90` data repeats one generic question with different answers. The canonical 400-row retain evaluation includes one of the two conflicting rows. Primary TOFU metrics retain the official row; the sensitivity row below excludes it.",
        "",
        "| Model | Retain probability, primary → excluding collision | Retain ROUGE-L recall, primary → excluding collision | Retain truth-ratio score, primary → excluding collision |",
        "|---|---:|---:|---:|",
    ]
    for key in keys:
        rows = results[key]["per_example"]["retain90"]
        clean = [row for row in rows if not row["ambiguous_question_collision"]]
        all_summary = summarize_examples(rows)
        clean_summary = summarize_examples(clean)
        lines.append(
            f"| {results[key]['model']['label']} | {all_summary['mean_answer_probability']:.4f} → {clean_summary['mean_answer_probability']:.4f} | "
            f"{all_summary['mean_rougeL_recall']:.4f} → {clean_summary['mean_rougeL_recall']:.4f} | "
            f"{all_summary['mean_truth_ratio_score']:.4f} → {clean_summary['mean_truth_ratio_score']:.4f} |"
        )
    lines += [
        "",
        "## Interpretation limits",
        "",
        "- Confidence intervals resample the 20 inferred profile blocks for forget10, canonical retain, and holdout10. The JSON contains question-level outputs; real-author and world-fact rows have no explicit cluster IDs, so they are reported without profile-cluster intervals.",
        "- Inference uses FP16 + SDPA because Kaggle T4 does not provide the upstream BF16 + FlashAttention 2 runtime. Small numeric and generation differences from an upstream evaluation are possible.",
        "- The 3B and 8B checkpoints come from different Llama generations and training histories. Their paired differences are descriptive checkpoint comparisons, not causal evidence about parameter count.",
        "- A small/large ROUGE or likelihood value alone does not prove forgetting. The forget-quality p-value compares distributions with a retain-only oracle and is not a standalone proof of equivalence or privacy.",
        "- TOFU profiles are fictional. These results cannot establish deletion of real personal data or real-world privacy protection.",
        "- The earlier 6 Nov likelihood-only run used a custom retain sample; this report uses the canonical `retain_perturbed` evaluation set from the pinned benchmark revision.",
        "",
        "## Reproduction metadata",
        "",
        f"Dataset revision: `{DATASET_REVISION}`. Open-Unlearning code revision: `{UPSTREAM_REVISION}`. Bootstrap: {BOOTSTRAP_DRAWS:,} profile-cluster draws, seed {BOOTSTRAP_SEED}. Greedy generation: max {MAX_NEW_TOKENS} new tokens.",
        "",
    ]
    (output_dir / "official_metric_baseline_report.md").write_text("\n".join(lines), encoding="utf-8")

    compare = {}
    for split in split_labels:
        left = results["llama_3_2_3b_full"]["per_example"][split]
        right = results["llama_3_1_8b_full"]["per_example"][split]
        right_by_index = {row["source_row_index"]: row for row in right}
        if len(right_by_index) != len(right) or {row["source_row_index"] for row in left} != set(right_by_index):
            raise ValueError(f"Full checkpoints do not have matched source examples in {split}")
        item = {}
        for metric in ("answer_probability", "rougeL_recall", "truth_ratio", "truth_ratio_score", "normalized_option_probability"):
            pairs = [(row, right_by_index[row["source_row_index"]]) for row in left if metric in row and metric in right_by_index[row["source_row_index"]]]
            if not pairs:
                continue
            deltas = [float(b[metric]) - float(a[metric]) for a, b in pairs]
            metric_result = {"n_questions": len(pairs), "mean_8b_minus_3b": sum(deltas) / len(deltas)}
            grouped = collections.defaultdict(list)
            for (a, _), delta in zip(pairs, deltas):
                if a.get("profile_block") is not None:
                    grouped[int(a["profile_block"])].append(delta)
            if grouped:
                group_ids = list(grouped)
                rng = random.Random(BOOTSTRAP_SEED)
                estimates = []
                for _ in range(BOOTSTRAP_DRAWS):
                    sampled = [rng.choice(group_ids) for _ in group_ids]
                    vals = [value for group in sampled for value in grouped[group]]
                    estimates.append(sum(vals) / len(vals))
                estimates.sort()
                metric_result["profile_cluster_bootstrap_95pct_ci"] = [
                    estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
                    estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
                ]
            else:
                metric_result["interval"] = None
                metric_result["interval_note"] = "No explicit profile cluster IDs; no inferential interval reported."
            item[metric] = metric_result
        compare[split] = item
    compare["interpretation_limit"] = "Descriptive paired checkpoint contrasts only; model generations/training histories differ."
    (output_dir / "paired_full_checkpoint_comparison.json").write_text(json.dumps(compare, indent=2) + "\n", encoding="utf-8")


def encode_jobs(tokenizer, jobs: list[dict]) -> list[tuple[list[int], list[int]]]:
    return [encode_answer(tokenizer, job["question"], job["candidate_answer"]) for job in jobs]


def evaluate_checkpoint(
    key: str,
    output_dir: Path,
    eval_sets: dict[str, list[dict]],
    dataset_hashes: dict[str, str],
    score_batch_size: int,
    generation_batch_size: int,
    max_memory_per_gpu_gib: float,
    adapter_path: Path | None = None,
    result_key: str | None = None,
    adapter_seed: int | None = None,
) -> dict:
    from accelerate import __version__ as accelerate_version
    from transformers import __version__ as transformers_version

    spec = CHECKPOINTS[key]
    cache_root = Path(
        os.environ.get(
            "TOFU_MODEL_CACHE_ROOT",
            str(Path(tempfile.gettempdir()) / "tofu_official_eval_cache"),
        )
    )
    model_cache = cache_root / key
    model_cache.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], cache_dir=str(model_cache), use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if not tokenizer.is_fast:
        raise RuntimeError("Fast tokenizer required")

    score_jobs, generation_jobs = build_jobs(eval_sets, spec["role"])
    encoded_jobs = encode_jobs(tokenizer, score_jobs)
    prompt_ids = [encode_prompt(tokenizer, row["question"]) for row in generation_jobs]
    # Transformers 5.22's allocator warmup reserves a second buffer roughly
    # the size of the 8B model shard. On T4x2 this OOMs before weights load.
    # Disable only this load-time optimization for 8B; checkpoint weights still
    # load normally. Keep the library default for the smaller 3B model.
    allocator_warmup_strategy = "library default"
    if spec["family"] == "llama_3_1_8b":
        import transformers.modeling_utils as modeling_utils

        if not hasattr(modeling_utils, "caching_allocator_warmup"):
            raise RuntimeError("Pinned Transformers version lacks the expected allocator warmup hook")
        modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None
        allocator_warmup_strategy = "disabled for 8B to avoid duplicate preallocation; weights load normally"

    memory_limit = f"{max_memory_per_gpu_gib:g}GiB"
    # With only two T4s, balanced_low_0 put almost all 8B decoder blocks on
    # GPU 1 and caused CPU offload. Even balancing leaves ample generation
    # headroom on both GPUs at the configured 13 GiB cap.
    print(f"Loading {spec['label']} with balanced; max_memory={memory_limit} per T4", flush=True)
    model = AutoModelForCausalLM.from_pretrained(
        spec["repo_id"],
        revision=spec["revision"],
        cache_dir=str(model_cache),
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="balanced",
        max_memory={0: memory_limit, 1: memory_limit, "cpu": "40GiB"},
        attn_implementation="sdpa",
    )
    device_map = getattr(model, "hf_device_map", {})
    offloaded_devices = sorted({str(device) for device in device_map.values() if str(device).lower() in {"cpu", "disk"}})
    print({"loaded_model": spec["label"], "hf_device_map": device_map, "offloaded_devices": offloaded_devices}, flush=True)
    if spec["family"] == "llama_3_1_8b" and offloaded_devices:
        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise RuntimeError(f"8B placement used CPU/disk offload {offloaded_devices}; refusing a slow, unbounded evaluation")
    result_spec = dict(spec)
    if adapter_path is not None:
        from peft import PeftModel

        if not (adapter_path / "adapter_config.json").is_file():
            raise FileNotFoundError(f"LoRA adapter config missing from {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
        result_spec["label"] = f"{spec['label']} NPO LoRA seed {adapter_seed}"
        result_spec["adapter"] = {
            "method": "NPO + retain NLL, LoRA adapter",
            "seed": adapter_seed,
            "adapter_dir_name": adapter_path.name,
            "files_sha256": {
                p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                for p in sorted(adapter_path.iterdir()) if p.is_file()
            },
        }
    model.eval()
    model.config.use_cache = False

    # A tiny non-research prompt tests generation before the full dataset pass.
    smoke_ids = encode_prompt(tokenizer, "Reply with one short greeting.")
    smoke_tensor = torch.tensor([smoke_ids], dtype=torch.long, device=model.get_input_embeddings().weight.device)
    smoke_started = time.monotonic()
    model.config.use_cache = True
    with torch.inference_mode():
        smoke_output = model.generate(
            input_ids=smoke_tensor,
            attention_mask=torch.ones_like(smoke_tensor),
            do_sample=False,
            max_new_tokens=4,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    model.config.use_cache = False
    smoke_new_tokens = int(smoke_output.shape[1] - smoke_tensor.shape[1])
    if smoke_new_tokens < 1:
        raise RuntimeError("GPU placement smoke generated no tokens; refusing full evaluation")
    placement_smoke = {
        "status": "PASS",
        "new_tokens": smoke_new_tokens,
        "elapsed_seconds": round(time.monotonic() - smoke_started, 2),
        "research_rows_used": 0,
    }
    print({"placement_smoke": placement_smoke}, flush=True)

    print(f"Starting answer scoring: {len(encoded_jobs)} candidates", flush=True)
    score_values = score_answers(model, encoded_jobs, int(tokenizer.pad_token_id), score_batch_size)
    if len(score_values) != len(score_jobs):
        raise RuntimeError("Scored row count does not equal job count")
    score_map = score_rows_by_key(score_jobs, score_values)
    generations = {}
    if spec["role"] == "full":
        print(f"Starting greedy generation: {len(prompt_ids)} prompts", flush=True)
        generated_texts = generate_answers(model, tokenizer, prompt_ids, generation_batch_size)
        if len(generated_texts) != len(generation_jobs):
            raise RuntimeError("Generation row count does not equal prompt count")
        generations = {
            (row["split"], row["eval_row_index"]): text
            for row, text in zip(generation_jobs, generated_texts)
        }

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge.tokenizer = tokenizer
    splits_to_write = ("forget10",) if spec["role"] == "retain90_oracle" else tuple(eval_sets)
    metrics_by_split = {
        split: make_metrics(eval_sets[split], split, score_map, generations, tokenizer, rouge)
        for split in splits_to_write
    }
    result = {
        "status": "PASS_OFFICIAL_TOFU_ADAPTER_EVAL" if adapter_path is not None else "PASS_OFFICIAL_TOFU_BASELINE",
        "model": result_spec,
        "evaluation": {
            "dataset_id": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "dataset_sha256_by_file": dataset_hashes,
            "open_unlearning_revision": UPSTREAM_REVISION,
            "profile_block_size": PROFILE_QA_COUNT,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "bootstrap_draws": BOOTSTRAP_DRAWS,
            "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "use_cache": True},
            "declared_max_input_length": MAX_LENGTH,
            "input_truncation": False,
            "max_scored_sequence_tokens": max((len(tokens) for tokens, _ in encoded_jobs), default=0),
            "scored_sequences_over_declared_max_length": sum(len(tokens) > MAX_LENGTH for tokens, _ in encoded_jobs),
            "max_generation_prompt_tokens": max((len(tokens) for tokens in prompt_ids), default=0),
            "canonical_retain_eval_rows": len(eval_sets["retain90"]),
            "canonical_retain_profile_blocks": sorted({row["profile_block"] for row in eval_sets["retain90"]}),
            "score_jobs": len(score_jobs),
            "generation_jobs": len(generation_jobs),
            "metric_implementation": "Pinned Open-Unlearning probability, ROUGE-L recall, Truth Ratio, normalized option probability, model-utility harmonic mean, and KS forget-quality definitions; see README for exact conventions.",
        },
        "summary": {split: summarize_examples(rows) for split, rows in metrics_by_split.items()},
        "per_example": {split: rows for split, rows in metrics_by_split.items()},
        "model_utility": model_utility(metrics_by_split) if spec["role"] == "full" else None,
        "runtime": {
            "torch_version": torch.__version__,
            "transformers_version": transformers_version,
            "accelerate_version": accelerate_version,
            "rouge_score_version": __import__("importlib.metadata", fromlist=["version"]).version("rouge-score"),
            "scipy_version": __import__("scipy").__version__,
            "inference_dtype": "float16",
            "max_memory_per_gpu_gib": max_memory_per_gpu_gib,
            "device_map_strategy": "balanced",
            "hf_device_map": {str(k): str(v) for k, v in device_map.items()},
            "cpu_offload_used": bool(offloaded_devices),
            "allocator_warmup_strategy": allocator_warmup_strategy,
            "placement_smoke": placement_smoke,
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
            "elapsed_seconds": round(time.monotonic() - started, 2),
            "training_started": False,
            "research_adapter_created": adapter_path is not None,
        },
    }
    result_key = result_key or key
    result_path = output_dir / f"{result_key}_official_eval.json"
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    score_path = output_dir / f"{result_key}_examples.jsonl"
    with score_path.open("w", encoding="utf-8") as f:
        for rows in metrics_by_split.values():
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    del model, tokenizer, encoded_jobs, prompt_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(model_cache, ignore_errors=True)
    return result


def add_forget_quality(results_dir: Path, full_key: str, oracle_key: str) -> None:
    full_path = results_dir / f"{full_key}_official_eval.json"
    oracle_path = results_dir / f"{oracle_key}_official_eval.json"
    if not full_path.exists() or not oracle_path.exists():
        return
    full = json.loads(full_path.read_text(encoding="utf-8"))
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    full_rows = full["per_example"]["forget10"]
    oracle_rows = oracle["per_example"]["forget10"]
    if [(r["source_row_index"], r["question_sha256"]) for r in full_rows] != [
        (r["source_row_index"], r["question_sha256"]) for r in oracle_rows
    ]:
        raise ValueError("Full and retain90 oracle forget rows are not aligned")
    stat = ks_2samp(
        [1.0 / (row["truth_ratio"] + EPSILON) for row in full_rows],
        [1.0 / (row["truth_ratio"] + EPSILON) for row in oracle_rows],
    )
    full["forget_quality_vs_retain90_oracle"] = {
        "metric": "KS p-value on inverse forget truth-ratio distributions; monotonic inversion leaves the two-sample KS statistic and p-value unchanged",
        "n_full_checkpoint": len(full_rows),
        "n_retain90_oracle": len(oracle_rows),
        "ks_statistic": float(stat.statistic),
        "p_value": float(stat.pvalue),
        "interpretation": "A larger p-value means these two empirical distributions are less distinguishable by this KS test; it is not proof of successful unlearning.",
    }
    full_path.write_text(json.dumps(full, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/tofu_official_eval"))
    parser.add_argument("--models", default=",".join(CHECKPOINTS))
    parser.add_argument("--score-batch-size", type=int, default=2)
    parser.add_argument("--generation-batch-size", type=int, default=2)
    parser.add_argument("--max-memory-per-gpu-gib", type=float, default=13.0)
    parser.add_argument("--resume-existing", action="store_true", help="Reuse passing, pinned result files already in output-dir; final verifier checks every row and metric.")
    args = parser.parse_args()
    selected = [x.strip() for x in args.models.split(",") if x.strip()]
    if not selected or set(selected) - set(CHECKPOINTS):
        parser.error(f"models must be selected from {sorted(CHECKPOINTS)}")
    if args.max_memory_per_gpu_gib <= 0 or args.max_memory_per_gpu_gib >= 14:
        parser.error("max-memory-per-gpu-gib must be >0 and <14 to leave T4 runtime headroom")
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Requires GPU T4 x2; stopped before loading model weights")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    data, hashes = load_data()
    eval_sets = build_eval_sets(data)
    status_path = args.output_dir / "official_eval_status.json"
    reused_results = {}
    if args.resume_existing and status_path.exists():
        previous = json.loads(status_path.read_text(encoding="utf-8"))
        if previous.get("dataset_hashes") != hashes:
            raise RuntimeError("Cannot resume: existing receipt has different pinned dataset hashes")
        for key, entry in previous.get("model_results", {}).items():
            if entry.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
                continue
            result_path = args.output_dir / f"{key}_official_eval.json"
            examples_path = args.output_dir / f"{key}_examples.jsonl"
            if key not in CHECKPOINTS or not result_path.is_file() or not examples_path.is_file():
                raise RuntimeError(f"Cannot resume: incomplete successful receipt for {key}")
            prior_result = json.loads(result_path.read_text(encoding="utf-8"))
            spec = CHECKPOINTS[key]
            if (
                prior_result.get("status") != "PASS_OFFICIAL_TOFU_BASELINE"
                or prior_result.get("model", {}).get("repo_id") != spec["repo_id"]
                or prior_result.get("model", {}).get("revision") != spec["revision"]
                or prior_result.get("model", {}).get("role") != spec["role"]
                or prior_result.get("evaluation", {}).get("dataset_sha256_by_file") != hashes
            ):
                raise RuntimeError(f"Cannot resume: pinned model/data metadata mismatch for {key}")
            reused_results[key] = entry
        if set(selected) & set(reused_results):
            raise RuntimeError("Requested model is already marked successful; remove it from --models or start a clean output directory")

    for key in selected:
        if CHECKPOINTS[key]["role"] == "full":
            oracle_key = key.replace("_full", "_retain90")
            if oracle_key not in selected and oracle_key not in reused_results:
                parser.error(f"{key} requires its matching retain90 oracle in --models or a verified resumed output")

    status = {
        "status": "RUNNING",
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "selected_models": selected,
        "reused_models": sorted(reused_results),
        "dataset_hashes": hashes,
        "model_results": reused_results,
        "training_started": False,
    }
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    for key in selected:
        try:
            result = evaluate_checkpoint(
                key,
                args.output_dir,
                eval_sets,
                hashes,
                args.score_batch_size,
                args.generation_batch_size,
                args.max_memory_per_gpu_gib,
            )
            status["model_results"][key] = {
                "status": result["status"],
                "result_file": f"{key}_official_eval.json",
                "examples_file": f"{key}_examples.jsonl",
                "runtime_seconds": result["runtime"]["elapsed_seconds"],
            }
        except Exception as error:
            shutil.rmtree(Path(tempfile.gettempdir()) / "tofu_official_eval_cache" / key, ignore_errors=True)
            status["status"] = "ERROR"
            status["error"] = f"{type(error).__name__}: {error}"
            status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            raise
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"{key}: PASS_OFFICIAL_TOFU_BASELINE", flush=True)
    for full_key, oracle_key in (
        ("llama_3_2_3b_full", "llama_3_2_3b_retain90"),
        ("llama_3_1_8b_full", "llama_3_1_8b_retain90"),
    ):
        if full_key in status["model_results"] and oracle_key in status["model_results"]:
            add_forget_quality(args.output_dir, full_key, oracle_key)
    complete_keys = set(CHECKPOINTS)
    if complete_keys <= set(status["model_results"]):
        write_reports(args.output_dir)
    else:
        status["status"] = "PARTIAL"
        status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"Partial results saved under {args.output_dir}; missing {sorted(complete_keys - set(status['model_results']))}")
        return
    status["status"] = "COMPLETE"
    status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Results saved under {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/verify_tofu_official_eval.py
#!/usr/bin/env python3
"""Independently verify canonical TOFU baseline outputs and derived metrics."""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import random
from pathlib import Path

from rouge_score import rouge_scorer
from scipy.stats import ks_2samp


EXPECTED = {
    "llama_3_2_3b_full": ("open-unlearning/tofu_Llama-3.2-3B-Instruct_full", "24f31ca19f6966dcb6f6b29abc511cce71222d4a", "full"),
    "llama_3_2_3b_retain90": ("open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90", "2c35819e9a731d096df9abd6efbcc361870c5333", "retain90_oracle"),
    "llama_3_1_8b_full": ("open-unlearning/tofu_Llama-3.1-8B-Instruct_full", "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e", "full"),
    "llama_3_1_8b_retain90": ("open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90", "f02956a9f25a18f38ad7808fbefb2f957347273e", "retain90_oracle"),
}
SPLIT_COUNTS = {
    "forget10": (400, 20),
    "retain90": (400, 20),
    "holdout10": (400, 20),
    "real_authors": (100, None),
    "world_facts": (117, None),
}
EPS = 1e-10
N_BOOTSTRAP = 5000
SEED = 42
EVAL_DATA_FILES = {
    "forget10.json",
    "forget10_perturbed.json",
    "retain90.json",
    "retain_perturbed.json",
    "holdout10.json",
    "real_authors_perturbed.json",
    "world_facts_perturbed.json",
}
METRIC_FIELDS = (
    "answer_probability",
    "rougeL_recall",
    "truth_ratio",
    "truth_ratio_score",
    "normalized_option_probability",
)


def bootstrap_ci(rows: list[dict], field: str) -> list[float] | None:
    grouped = collections.defaultdict(list)
    for row in rows:
        if row.get("profile_block") is not None and row.get(field) is not None:
            grouped[int(row["profile_block"])].append(float(row[field]))
    if not grouped:
        return None
    groups = list(grouped.values())
    rng = random.Random(SEED)
    estimates = []
    for _ in range(N_BOOTSTRAP):
        sample = [rng.choice(groups) for _ in groups]
        values = [x for block in sample for x in block]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return [estimates[int(.025 * (N_BOOTSTRAP - 1))], estimates[int(.975 * (N_BOOTSTRAP - 1))]]


def mean(rows: list[dict], field: str) -> float | None:
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None


def equal(a, b, field: str, problems: list[str], tolerance: float = 1e-10) -> None:
    if a is None or b is None:
        problems.append(f"{field}: missing numeric value (expected {b}, found {a})")
        return
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        problems.append(f"{field}: nonnumeric value (expected {b!r}, found {a!r})")
        return
    if not math.isfinite(float(a)) or not math.isfinite(float(b)):
        problems.append(f"{field}: non-finite value (expected {b!r}, found {a!r})")
        return
    if not math.isclose(float(a), float(b), rel_tol=tolerance, abs_tol=tolerance):
        problems.append(f"{field}: expected {b}, found {a}")


def verify_rows(key: str, result: dict, audit: dict) -> tuple[list[str], dict]:
    problems = []
    repo, revision, role = EXPECTED[key]
    model = result.get("model", {})
    if result.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
        problems.append("unexpected result status")
    if (model.get("repo_id"), model.get("revision"), model.get("role")) != (repo, revision, role):
        problems.append("model repository/revision/role does not match pinned protocol")
    runtime = result.get("runtime", {})
    if runtime.get("training_started") is not False or runtime.get("research_adapter_created") is not False:
        problems.append("runtime receipt does not confirm inference-only execution")
    if runtime.get("gpu_names", []).count("Tesla T4") < 2:
        problems.append("receipt does not show two Tesla T4 GPUs")
    if key.startswith("llama_3_1_8b"):
        if runtime.get("device_map_strategy") != "balanced":
            problems.append("8B receipt does not use the pinned balanced device map")
        if runtime.get("cpu_offload_used") is not False:
            problems.append("8B receipt does not confirm GPU-only model placement")
        if runtime.get("placement_smoke", {}).get("status") != "PASS":
            problems.append("8B receipt lacks a passing pre-evaluation generation smoke")
        if not runtime.get("hf_device_map"):
            problems.append("8B receipt lacks the resolved Hugging Face device map")
    if result.get("evaluation", {}).get("input_truncation") is not False:
        problems.append("unexpected input truncation policy")
    expected_files = audit["data_audit"]["files"]
    recorded_hashes = result.get("evaluation", {}).get("dataset_sha256_by_file", {})
    if set(recorded_hashes) != EVAL_DATA_FILES:
        problems.append("dataset checksum file set does not match the pinned evaluation inputs")
    for filename, digest in recorded_hashes.items():
        if filename not in expected_files or expected_files[filename]["sha256"] != digest:
            problems.append(f"dataset checksum mismatch for {filename}")
    if result.get("evaluation", {}).get("open_unlearning_revision") != "2d98027677f3468d52b6baa3fc4f2b9f2dc08820":
        problems.append("Open-Unlearning source revision does not match the pinned protocol")

    expected_splits = ("forget10",) if role == "retain90_oracle" else tuple(SPLIT_COUNTS)
    all_examples = result.get("per_example", {})
    if set(all_examples) != set(expected_splits):
        problems.append(f"unexpected evaluated split keys: {sorted(all_examples)}")
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    recomputed_summary = {}
    for split in expected_splits:
        rows = all_examples.get(split, [])
        expected_n, expected_groups = SPLIT_COUNTS[split]
        if len(rows) != expected_n:
            problems.append(f"{split}: expected {expected_n} examples, found {len(rows)}")
        source_indices = [row.get("source_row_index") for row in rows]
        if len(set(source_indices)) != len(source_indices):
            problems.append(f"{split}: duplicate source row index")
        if split in {"forget10", "holdout10", "retain90"} and source_indices != list(range(expected_n)):
            problems.append(f"{split}: source row indices do not exactly cover the pinned ordered evaluation rows")
        if split in {"real_authors", "world_facts"} and source_indices != list(range(expected_n)):
            problems.append(f"{split}: source row indices do not exactly cover the pinned ordered utility rows")
        groups = {row.get("profile_block") for row in rows if row.get("profile_block") is not None}
        if expected_groups is not None and len(groups) != expected_groups:
            problems.append(f"{split}: expected {expected_groups} profile groups, found {len(groups)}")
        if expected_groups is not None and groups != set(range(expected_groups)):
            problems.append(f"{split}: profile group IDs are not the expected contiguous blocks")
        for i, row in enumerate(rows):
            p = row.get("answer_probability")
            loss = row.get("answer_token_loss")
            if not isinstance(p, (int, float)) or not math.isfinite(p) or not 0 <= p <= 1:
                problems.append(f"{split} row {i}: invalid answer probability")
            if not isinstance(loss, (int, float)) or not math.isfinite(loss) or loss < 0:
                problems.append(f"{split} row {i}: invalid answer loss")
            else:
                equal(p, math.exp(-float(loss)), f"{split} row {i} exp(-answer loss)", problems)
            if role == "full":
                generation = row.get("generation")
                ref = row.get("reference_answer")
                if not isinstance(generation, str) or not isinstance(ref, str):
                    problems.append(f"{split} row {i}: missing generated text/reference")
                else:
                    expected_rouge = rouge.score(ref, generation)["rougeL"].recall
                    equal(row.get("rougeL_recall"), expected_rouge, f"{split} row {i} ROUGE-L recall", problems)
            if "paraphrased_answer_token_loss" in row:
                para_loss = float(row["paraphrased_answer_token_loss"])
                wrong_losses = row.get("perturbed_answer_token_losses", [])
                if not wrong_losses:
                    problems.append(f"{split} row {i}: truth ratio has no perturbed-answer losses")
                else:
                    wrong_geo = math.exp(-sum(map(float, wrong_losses)) / len(wrong_losses))
                    correct_p = math.exp(-para_loss)
                    ratio = wrong_geo / (correct_p + EPS)
                    equal(row.get("truth_ratio"), ratio, f"{split} row {i} truth ratio", problems)
                    if split == "forget10":
                        score = min(ratio, 1.0 / (ratio + EPS))
                    else:
                        score = max(0.0, 1.0 - ratio)
                    equal(row.get("truth_ratio_score"), score, f"{split} row {i} truth-ratio score", problems)
            elif "perturbed_answer_token_losses" in row:
                wrong_losses = row["perturbed_answer_token_losses"]
                wrong_probs = row.get("perturbed_answer_probabilities", [])
                if len(wrong_losses) != len(wrong_probs) or not wrong_losses:
                    problems.append(f"{split} row {i}: wrong-option losses/probabilities are inconsistent")
                else:
                    for j, (wrong_loss, wrong_prob) in enumerate(zip(wrong_losses, wrong_probs)):
                        equal(wrong_prob, math.exp(-float(wrong_loss)), f"{split} row {i} wrong option {j} probability", problems)
                    wrong_geo = math.exp(-sum(map(float, wrong_losses)) / len(wrong_losses))
                    ratio = wrong_geo / (float(p) + EPS)
                    equal(row.get("truth_ratio"), ratio, f"{split} row {i} truth ratio", problems)
                    equal(row.get("truth_ratio_score"), max(0, 1.0 - ratio), f"{split} row {i} truth-ratio score", problems)
                    normalized = float(p) / (float(p) + sum(map(float, wrong_probs)) + EPS)
                    equal(row.get("normalized_option_probability"), normalized, f"{split} row {i} normalized option probability", problems)
            for field in ("truth_ratio", "truth_ratio_score", "normalized_option_probability", "rougeL_recall"):
                value = row.get(field)
                if value is not None and (not math.isfinite(float(value)) or float(value) < 0):
                    problems.append(f"{split} row {i}: invalid {field}")
                if field == "rougeL_recall" and value is not None and float(value) > 1:
                    problems.append(f"{split} row {i}: ROUGE-L recall is outside [0,1]")

        summary = {"n_questions": len(rows), "n_profile_blocks": len(groups)}
        for field in METRIC_FIELDS:
            value = mean(rows, field)
            if value is None:
                continue
            summary[f"mean_{field}"] = value
            ci = bootstrap_ci(rows, field)
            if ci is not None:
                summary[f"profile_cluster_bootstrap_95pct_ci_{field}"] = ci
        recorded = result.get("summary", {}).get(split, {})
        if summary.keys() != recorded.keys():
            problems.append(f"{split}: summary fields mismatch; recomputed={sorted(summary)}, recorded={sorted(recorded)}")
        for field, value in summary.items():
            if isinstance(value, list):
                if len(recorded.get(field, [])) != len(value):
                    problems.append(f"{split}: {field} length mismatch")
                else:
                    for j, (a, b) in enumerate(zip(recorded[field], value)):
                        equal(a, b, f"{split} {field}[{j}]", problems)
            elif isinstance(value, float):
                equal(recorded.get(field), value, f"{split} {field}", problems)
            elif recorded.get(field) != value:
                problems.append(f"{split}: {field} mismatch")
        recomputed_summary[split] = summary

    if role == "full":
        retain = all_examples["retain90"]
        if [row["source_row_index"] for row in retain] != list(range(400)):
            problems.append("canonical retain evaluation rows are not the pinned first 20 complete profiles")
        collision_rows = [row for row in retain if row.get("ambiguous_question_collision")]
        if len(collision_rows) != 1 or collision_rows[0].get("source_row_index") != 100:
            problems.append("canonical retain evaluation must contain exactly one side of the known collision")
        utility = result.get("model_utility", {})
        component_values = {
            "retain_probability": mean(all_examples["retain90"], "answer_probability"),
            "retain_rougeL_recall": mean(all_examples["retain90"], "rougeL_recall"),
            "retain_truth_ratio_score": mean(all_examples["retain90"], "truth_ratio_score"),
            "real_authors_normalized_option_probability": mean(all_examples["real_authors"], "normalized_option_probability"),
            "real_authors_rougeL_recall": mean(all_examples["real_authors"], "rougeL_recall"),
            "real_authors_truth_ratio_score": mean(all_examples["real_authors"], "truth_ratio_score"),
            "world_facts_normalized_option_probability": mean(all_examples["world_facts"], "normalized_option_probability"),
            "world_facts_rougeL_recall": mean(all_examples["world_facts"], "rougeL_recall"),
            "world_facts_truth_ratio_score": mean(all_examples["world_facts"], "truth_ratio_score"),
        }
        if utility.get("components", {}).keys() != component_values.keys():
            problems.append("model-utility component keys do not match official nine metrics")
        for name, value in component_values.items():
            equal(utility.get("components", {}).get(name), value, f"model utility {name}", problems)
        expected_hm = 0.0 if any(value <= 0 for value in component_values.values()) else 9 / sum(1 / x for x in component_values.values())
        equal(utility.get("harmonic_mean"), expected_hm, "TOFU model-utility harmonic mean", problems)

    return problems, recomputed_summary


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("result_dir", type=Path)
    parser.add_argument("--audit", type=Path, required=True)
    parser.add_argument(
        "--allow-partial",
        action="store_true",
        help="Verify available checkpoints and report PARTIAL when other pinned checkpoints are absent.",
    )
    args = parser.parse_args()
    audit = json.loads(args.audit.read_text(encoding="utf-8"))
    problems = []
    run_status = None
    raw_run_status = None
    run_error = None
    if not audit.get("data_audit", {}).get("checks", {}).get("structural_checks_pass"):
        problems.append("pinned dataset audit did not pass")
    status_file = args.result_dir / "official_eval_status.json"
    if status_file.exists():
        status = json.loads(status_file.read_text(encoding="utf-8"))
        run_status = status.get("status")
        raw_run_status = status.get("raw_kaggle_terminal_status", status.get("raw_run_status"))
        run_error = status.get("error") or status.get("raw_kaggle_error")
        acceptable_statuses = {"COMPLETE", "PARTIAL", "ERROR"} if args.allow_partial else {"COMPLETE"}
        if status.get("status") not in acceptable_statuses:
            problems.append(f"run status is {status.get('status')}, expected COMPLETE")
    else:
        problems.append("missing official_eval_status.json")

    model_results = {}
    recomputed = {}
    for key in EXPECTED:
        path = args.result_dir / f"{key}_official_eval.json"
        if not path.exists():
            model_results[key] = {"status": "MISSING", "problems": [f"missing {path.name}"]}
            if not args.allow_partial:
                problems.append(f"{key}: missing {path.name}")
            continue
        result = json.loads(path.read_text(encoding="utf-8"))
        local_problems, local_summary = verify_rows(key, result, audit)
        model_results[key] = {
            "status": "PASS" if not local_problems else "FAIL",
            "problems": local_problems,
            "model": result.get("model"),
            "summary": local_summary,
            "runtime": result.get("runtime"),
        }
        recomputed[key] = result
        problems.extend(f"{key}: {item}" for item in local_problems)

    for full_key, oracle_key in (
        ("llama_3_2_3b_full", "llama_3_2_3b_retain90"),
        ("llama_3_1_8b_full", "llama_3_1_8b_retain90"),
    ):
        if full_key in recomputed and oracle_key in recomputed:
            full = recomputed[full_key]
            oracle = recomputed[oracle_key]
            full_rows = full["per_example"]["forget10"]
            oracle_rows = oracle["per_example"]["forget10"]
            if [(r["source_row_index"], r["question_sha256"]) for r in full_rows] != [(r["source_row_index"], r["question_sha256"]) for r in oracle_rows]:
                problems.append(f"{full_key}: forget rows do not align with retain oracle")
            else:
                stat = ks_2samp(
                    [1 / (r["truth_ratio"] + EPS) for r in full_rows],
                    [1 / (r["truth_ratio"] + EPS) for r in oracle_rows],
                )
                recorded = full.get("forget_quality_vs_retain90_oracle", {})
                equal(recorded.get("ks_statistic"), stat.statistic, f"{full_key} forget-quality KS statistic", problems)
                equal(recorded.get("p_value"), stat.pvalue, f"{full_key} forget-quality p-value", problems)
    present_statuses = [x["status"] for x in model_results.values() if x["status"] != "MISSING"]
    missing_models = [key for key, value in model_results.items() if value["status"] == "MISSING"]
    if problems or any(value != "PASS" for value in present_statuses):
        overall = "FAIL"
    elif missing_models and args.allow_partial:
        overall = "PARTIAL"
    elif missing_models:
        overall = "FAIL"
    else:
        overall = "PASS"
    verification = {
        "overall_status": overall,
        "run_status": run_status,
        "raw_run_status": raw_run_status,
        "run_error": run_error,
        "audit_status": "PASS" if audit.get("data_audit", {}).get("checks", {}).get("structural_checks_pass") else "FAIL",
        "missing_models": missing_models,
        "problems": problems,
        "models": model_results,
    }
    (args.result_dir / "official_eval_verification.json").write_text(json.dumps(verification, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    lines = ["# TOFU official-metric baseline verification", "", f"Overall: **{overall}**", f"Reconciled run status: **{run_status}**"]
    if raw_run_status:
        lines.append(f"Raw Kaggle terminal status: **{raw_run_status}**")
    if run_error:
        lines.append(f"Raw run error: `{run_error}`")
    lines.append("")
    for key, item in model_results.items():
        lines.append(f"## {key}: {item['status']}")
        lines.append("")
        for message in item.get("problems", []):
            lines.append(f"- {message}")
        for split, summary in item.get("summary", {}).items():
            lines.append(f"- {split}: {summary.get('n_questions')} questions, {summary.get('n_profile_blocks')} profile blocks")
        lines.append("")
    for message in problems:
        if not any(message in item.get("problems", []) for item in model_results.values()):
            lines.append(f"- {message}")
    (args.result_dir / "official_eval_verification.md").write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps(verification, indent=2))
    if overall == "FAIL":
        raise SystemExit(1)
    if overall == "PARTIAL" and not args.allow_partial:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/npo_objective.py
"""Minimal NPO math shared by training and the CPU objective-equivalence check."""

import torch
import torch.nn.functional as F


IGNORE_INDEX = -100


def sequence_nll(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return summed answer NLL and scored-token count per sequence."""
    shift_logits = logits[..., :-1, :].contiguous().float()
    shift_labels = labels[..., 1:].contiguous().to(logits.device)
    losses = F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.shape[-1]),
        shift_labels.reshape(-1),
        ignore_index=IGNORE_INDEX,
        reduction="none",
    ).reshape(shift_labels.shape)
    counts = (shift_labels != IGNORE_INDEX).sum(dim=-1)
    if torch.any(counts == 0):
        raise ValueError("An NPO example has no assistant answer tokens")
    return losses.sum(dim=-1), counts


def objective_from_nlls(current_nll: torch.Tensor, reference_nll: torch.Tensor, beta: float) -> torch.Tensor:
    """Pinned Open-Unlearning NPO loss for sequence-summed answer NLL."""
    # Upstream computes -2/beta * log(sigmoid(-beta * log(pi_theta/pi_ref))).
    # Since log(pi_theta/pi_ref) = reference_nll - current_nll, this is
    # beta * (current_nll - reference_nll) inside the sigmoid.
    return (-2.0 / beta * F.logsigmoid(beta * (current_nll - reference_nll))).mean()


In [ ]:
%%writefile /kaggle/working/verify_npo_objective.py
#!/usr/bin/env python3
"""Check NPO loss/value/gradient equivalence against pinned Open-Unlearning."""

from __future__ import annotations

import argparse
import copy
import hashlib
import importlib.util
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import torch
import torch.nn.functional as F

from npo_objective import IGNORE_INDEX, objective_from_nlls, sequence_nll


class TinyCausalLM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(13, 7)
        self.head = torch.nn.Linear(7, 13, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.head(self.embedding(input_ids))
        shift_logits = logits[..., :-1, :].contiguous().float()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=IGNORE_INDEX,
            reduction="mean",
        )
        return SimpleNamespace(logits=logits, loss=loss)


def max_abs_grad_difference(left, right):
    deltas = []
    for p_left, p_right in zip(left.parameters(), right.parameters()):
        if p_left.grad is None or p_right.grad is None:
            if p_left.grad is not p_right.grad:
                return float("inf")
            continue
        deltas.append(float((p_left.grad - p_right.grad).abs().max()))
    return max(deltas, default=0.0)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--upstream-src", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise RuntimeError("NPO pilot protocol is not registered")
    if protocol.get("objective", {}).get("source") != "locuslab/open-unlearning@2d98027677f3468d52b6baa3fc4f2b9f2dc08820":
        raise RuntimeError("Protocol does not pin the expected Open-Unlearning implementation")
    upstream_src = args.upstream_src.resolve()
    upstream_utils = upstream_src / "trainer" / "utils.py"
    if not upstream_utils.is_file():
        raise FileNotFoundError(upstream_utils)
    spec = importlib.util.spec_from_file_location("pinned_open_unlearning_utils", upstream_utils)
    upstream = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(upstream)
    compute_dpo_loss = upstream.compute_dpo_loss

    torch.manual_seed(42)
    beta = float(protocol["objective"]["beta"])
    inputs = {
        "input_ids": torch.tensor([[1, 2, 3, 4, 5], [2, 3, 4, 6, 7]]),
        "attention_mask": torch.ones((2, 5), dtype=torch.long),
        "labels": torch.tensor([[-100, -100, 3, 4, 5], [-100, 3, 4, 6, 7]]),
    }
    base = TinyCausalLM()
    model_upstream = copy.deepcopy(base)
    model_candidate = copy.deepcopy(base)
    ref_upstream = copy.deepcopy(base)
    ref_candidate = copy.deepcopy(base)

    upstream_loss, _ = compute_dpo_loss(
        model=model_upstream,
        ref_model=ref_upstream,
        win_inputs=None,
        lose_inputs=inputs,
        beta=beta,
    )
    upstream_loss.backward()

    current_logits = model_candidate(**inputs).logits
    current_nll, _ = sequence_nll(current_logits, inputs["labels"])
    with torch.no_grad():
        reference_logits = ref_candidate(**inputs).logits
        reference_nll, _ = sequence_nll(reference_logits, inputs["labels"])
    candidate_loss = objective_from_nlls(current_nll, reference_nll, beta)
    candidate_loss.backward()

    value_diff = abs(float(upstream_loss.detach()) - float(candidate_loss.detach()))
    grad_diff = max_abs_grad_difference(model_upstream, model_candidate)
    retain_reference_loss = model_candidate(**inputs).loss.detach()
    candidate_retain_loss = current_nll.detach().sum() / (inputs["labels"][..., 1:] != IGNORE_INDEX).sum()
    retain_diff = abs(float(retain_reference_loss) - float(candidate_retain_loss))

    model_combined = copy.deepcopy(base)
    reference_combined = copy.deepcopy(base)
    combined_forget, _ = compute_dpo_loss(
        model=model_combined,
        ref_model=reference_combined,
        win_inputs=None,
        lose_inputs=inputs,
        beta=beta,
    )
    combined_retain = model_combined(**inputs).loss
    (combined_forget + combined_retain).backward()

    model_sequential = copy.deepcopy(base)
    reference_sequential = copy.deepcopy(base)
    sequential_forget, _ = compute_dpo_loss(
        model=model_sequential,
        ref_model=reference_sequential,
        win_inputs=None,
        lose_inputs=inputs,
        beta=beta,
    )
    sequential_forget.backward()
    model_sequential(**inputs).loss.backward()
    sequential_grad_diff = max_abs_grad_difference(model_combined, model_sequential)
    result = {
        "status": "PASS" if value_diff <= 1e-6 and grad_diff <= 1e-6 and retain_diff <= 1e-6 and sequential_grad_diff <= 1e-6 else "FAIL",
        "upstream": protocol["objective"]["source"],
        "upstream_utils_sha256": hashlib.sha256(upstream_utils.read_bytes()).hexdigest(),
        "candidate_objective_sha256": hashlib.sha256(Path(__file__).with_name("npo_objective.py").read_bytes()).hexdigest(),
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "python_version": sys.version,
        "torch_version": torch.__version__,
        "npo_beta": beta,
        "upstream_loss": float(upstream_loss.detach()),
        "candidate_loss": float(candidate_loss.detach()),
        "absolute_loss_difference": value_diff,
        "maximum_parameter_gradient_difference": grad_diff,
        "retain_nll_loss_difference": retain_diff,
        "combined_vs_sequential_backward_max_gradient_difference": sequential_grad_diff,
        "examples": int(inputs["input_ids"].shape[0]),
        "loss_reduction": "sum token NLL per sequence; batch mean for NPO; token mean for retain NLL",
    }
    encoded = json.dumps(result, indent=2) + "\n"
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(encoded, encoding="utf-8")
    print(encoded, end="")
    if result["status"] != "PASS":
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/npo_lora_train.py
#!/usr/bin/env python3
"""Resource-bounded LoRA NPO for the pinned Open-Unlearning TOFU checkpoints.

The NPO and retain objectives follow Open-Unlearning at the pinned revision.
Only the LoRA adapter is trained; this is explicitly an adapter-based NPO
variant, not a reproduction of full-parameter NPO.
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

from tofu_official_eval import (
    CHECKPOINTS,
    DATASET_ID,
    DATASET_REVISION,
    UPSTREAM_REVISION,
    collate_scoring,
    encode_answer,
    get_bytes,
)
from npo_objective import objective_from_nlls, sequence_nll


METHOD = {
    "method": "NPO + retain NLL (LoRA adapter)",
    "beta": 0.1,
    "gamma_forget": 1.0,
    "alpha_retain": 1.0,
    "retain_loss_type": "NLL",
    "learning_rate": 1e-5,
    "weight_decay": 0.0,
    "epochs": 10,
    "max_length": 512,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "gradient_clip_norm": 1.0,
    "batching": "1 forget + 1 retain example per microbatch; effective paired batch 8",
    "optimizer": "torch AdamW, betas=(0.9,0.999), eps=1e-8",
    "scheduler": "linear; one epoch warmup",
    "precision": "FP16 frozen base checkpoint; FP32 LoRA parameters and loss; SDPA",
    "retain_sampling": "uniform with replacement, one retain row per forget row, as in pinned upstream anchor=forget wrapper",
}


MODEL_FAMILIES = {
    "3b": ("llama_3_2_3b_full", 2, 4),
    "8b": ("llama_3_1_8b_full", 1, 8),
}


def fetch_jsonl(filename: str) -> tuple[list[dict], str]:
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
    raw = get_bytes(url)
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    if not rows or any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
        raise ValueError(f"Invalid pinned TOFU train file: {filename}")
    return rows, hashlib.sha256(raw).hexdigest()


def collate(rows: list[tuple[list[int], list[int]]], pad_id: int, device: torch.device):
    input_ids, attention, labels = collate_scoring(rows, pad_id)
    return input_ids.to(device), attention.to(device), labels.to(device)


def encode_rows(tokenizer, rows: list[dict]) -> list[tuple[list[int], list[int]]]:
    encoded = [encode_answer(tokenizer, row["question"], row["answer"]) for row in rows]
    if any(len(ids) > METHOD["max_length"] for ids, _ in encoded):
        raise ValueError("Found a training sequence longer than the pinned 512-token limit; no silent truncation is allowed")
    return encoded


def reference_nlls(model, encoded, pad_id: int, device: torch.device, batch_size: int = 1) -> list[float]:
    model.eval()
    values: list[float] = []
    with torch.inference_mode():
        for start in range(0, len(encoded), batch_size):
            ids, attention, labels = collate(encoded[start:start + batch_size], pad_id, device)
            outputs = model(input_ids=ids, attention_mask=attention, use_cache=False)
            nll, _ = sequence_nll(outputs.logits, labels)
            values.extend(float(x) for x in nll.cpu().tolist())
    if len(values) != len(encoded) or not all(math.isfinite(x) and x >= 0 for x in values):
        raise RuntimeError("Reference NLL precomputation was incomplete or non-finite")
    return values


def train(args) -> dict:
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    objective_check = json.loads(args.objective_check.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise RuntimeError("NPO protocol is not marked preregistered; refusing to train")
    if objective_check.get("status") != "PASS":
        raise RuntimeError("Pinned-objective loss/gradient equivalence did not pass")
    expected_packages = {"transformers": "5.22.1", "accelerate": "1.15.0", "peft": "0.21.2", "scipy": "1.16.3"}
    actual_packages = {name: importlib.metadata.version(name) for name in expected_packages}
    if actual_packages != expected_packages:
        raise RuntimeError(f"Training package versions differ from the verified stack: {actual_packages}")
    torchao_spec = importlib.util.find_spec("torchao")
    if torchao_spec is not None:
        try:
            torchao_version = importlib.metadata.version("torchao")
        except importlib.metadata.PackageNotFoundError:
            torchao_version = "unknown-version"
        raise RuntimeError(
            f"Optional torchao {torchao_version} is installed. This unquantized LoRA run does not use torchao; "
            "remove it before loading the base model to keep PEFT on the ordinary Linear dispatcher."
        )
    if args.model != protocol.get("model_family") or args.seed != protocol.get("seed"):
        raise RuntimeError("Requested model/seed does not match the locked NPO diagnostic protocol")
    model_key, microbatch, accumulation = MODEL_FAMILIES[args.model]
    spec = CHECKPOINTS[model_key]
    locked_checkpoint = protocol.get("base_checkpoint", {})
    if (spec.get("repo_id"), spec.get("revision")) != (locked_checkpoint.get("repo_id"), locked_checkpoint.get("revision")):
        raise RuntimeError("Trainer checkpoint/revision differs from the preregistered protocol")
    if objective_check.get("upstream") != f"locuslab/open-unlearning@{UPSTREAM_REVISION}":
        raise RuntimeError("Objective receipt is for the wrong Open-Unlearning revision")
    if float(objective_check.get("npo_beta", -1)) != float(protocol["objective"]["beta"]):
        raise RuntimeError("Objective check beta differs from the preregistered protocol")
    locked = protocol["adaptation"]
    if (
        METHOD["lora_rank"] != locked["lora_rank"]
        or METHOD["lora_alpha"] != locked["lora_alpha"]
        or METHOD["lora_dropout"] != locked["lora_dropout"]
        or METHOD["lora_target_modules"] != locked["target_modules"]
        or METHOD["learning_rate"] != locked["learning_rate"]
        or METHOD["weight_decay"] != float(locked["optimizer"].split("weight_decay=")[-1].rstrip(")"))
        or METHOD["beta"] != protocol["objective"]["beta"]
        or METHOD["gamma_forget"] != protocol["objective"]["forget_weight_gamma"]
        or METHOD["alpha_retain"] != protocol["objective"]["retain_weight_alpha"]
        or METHOD["retain_loss_type"] != "NLL"
        or METHOD["precision"] != locked["precision"]
        or METHOD["scheduler"] != locked["scheduler"]
        or METHOD["epochs"] != locked["epochs"]
        or METHOD["max_length"] != locked["max_length"]
        or METHOD["gradient_clip_norm"] != locked["gradient_clip_norm"]
        or microbatch * accumulation != locked["effective_paired_batch_size"]
    ):
        raise RuntimeError("Trainer hyperparameters differ from the preregistered NPO-LoRA protocol")
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("NPO requires two GPUs; stopped before model download")
    gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    if gpu_names.count("Tesla T4") != 2:
        raise RuntimeError(f"Protocol requires Kaggle T4 x2, found {gpu_names}")

    output_dir: Path = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=False)
    run_type = "bounded_real_data_smoke" if args.smoke else "exploratory_single_seed"
    started = time.monotonic()
    receipt = {
        "status": "RUNNING",
        "run_type": run_type,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": spec,
        "dataset": {"repo": DATASET_ID, "revision": DATASET_REVISION},
        "upstream_code_revision": UPSTREAM_REVISION,
        "method": METHOD,
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "objective_equivalence": objective_check,
        "training_script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        "seed": args.seed,
        "gpu_names": gpu_names,
        "package_versions": {
            "torch": torch.__version__,
            "transformers": importlib.metadata.version("transformers"),
            "accelerate": importlib.metadata.version("accelerate"),
            "peft": importlib.metadata.version("peft"),
            "scipy": importlib.metadata.version("scipy"),
            "torchao": None,
        },
        "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
        "smoke_rows": 8 if args.smoke else None,
        "training_started": False,
        "adapter_saved": False,
    }
    receipt_path = output_dir / "training_receipt.json"
    trace_path = output_dir / "loss_trace.jsonl"
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")

    try:
        forget_rows, forget_sha = fetch_jsonl("forget10.json")
        retain_rows, retain_sha = fetch_jsonl("retain90.json")
        if len(forget_rows) != 400 or len(retain_rows) != 3600:
            raise ValueError(f"Unexpected pinned train counts: forget={len(forget_rows)}, retain={len(retain_rows)}")
        locked_files = protocol["dataset"]["training"]
        if forget_sha != locked_files["forget_file"]["sha256"] or retain_sha != locked_files["retain_file"]["sha256"]:
            raise ValueError("Fetched training data hash differs from the preregistered dataset revision and hash")
        receipt["dataset"].update({
            "forget10_rows": len(forget_rows),
            "retain90_rows": len(retain_rows),
            "forget10_sha256": forget_sha,
            "retain90_sha256": retain_sha,
        })

        cache_root = Path(os.environ.get("TOFU_MODEL_CACHE_ROOT", "/kaggle/working/model_cache"))
        checkpoint_cache = cache_root / model_key
        tokenizer = AutoTokenizer.from_pretrained(
            spec["repo_id"], revision=spec["revision"], cache_dir=str(checkpoint_cache), use_fast=True
        )
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        if not tokenizer.is_fast:
            raise RuntimeError("Pinned tokenizer must be fast")
        forget_encoded = encode_rows(tokenizer, forget_rows)
        retain_encoded = encode_rows(tokenizer, retain_rows)
        receipt["tokenization"] = {
            "max_length_observed": max(len(x[0]) for x in forget_encoded + retain_encoded),
            "truncated_examples": 0,
            "prompt_answer_prefix_check": "PASS for every sequence",
        }

        torch.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = False
        model = AutoModelForCausalLM.from_pretrained(
            spec["repo_id"],
            revision=spec["revision"],
            cache_dir=str(checkpoint_cache),
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="balanced",
            max_memory={0: "13GiB", 1: "13GiB", "cpu": "40GiB"},
            attn_implementation="sdpa",
        )
        device_map = getattr(model, "hf_device_map", {})
        offload = sorted({str(v) for v in device_map.values() if str(v).lower() in {"cpu", "disk"}})
        if offload:
            raise RuntimeError(f"Refusing training with CPU/disk offload: {offload}")
        input_device = model.get_input_embeddings().weight.device
        receipt["runtime"] = {
            "device_map": {str(k): str(v) for k, v in device_map.items()},
            "cpu_or_disk_offload": False,
            "input_device": str(input_device),
            "dtype": "float16",
            "attention": "sdpa",
        }

        if args.smoke:
            forget_encoded = forget_encoded[:8]
            # Keep the full retain pool in the real-data smoke; sample only the
            # first eight target steps to bound runtime and avoid a scored run.
            steps_to_run = 1
            batch_size = 1
            accumulation_steps = 1
            epochs = 1
        else:
            steps_to_run = None
            batch_size = microbatch
            accumulation_steps = accumulation
            epochs = METHOD["epochs"]

        receipt["training_started"] = True
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        ref_nll = reference_nlls(model, forget_encoded, int(tokenizer.pad_token_id), input_device)

        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        lora = LoraConfig(
            r=METHOD["lora_rank"],
            lora_alpha=METHOD["lora_alpha"],
            lora_dropout=METHOD["lora_dropout"],
            target_modules=METHOD["lora_target_modules"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora)
        trainable = [p for p in model.parameters() if p.requires_grad]
        if not trainable or any(p.dtype != torch.float32 for p in trainable):
            raise RuntimeError("Expected nonempty FP32 LoRA-only trainable parameters")
        for name, parameter in model.named_parameters():
            if parameter.requires_grad and "lora_" not in name:
                raise RuntimeError(f"Unexpected trainable base parameter: {name}")
        model.train()
        optimizer = torch.optim.AdamW(
            trainable,
            lr=METHOD["learning_rate"],
            betas=(0.9, 0.999),
            eps=1e-8,
            weight_decay=METHOD["weight_decay"],
        )

        effective_forget = 8
        if not args.smoke:
            batches_per_epoch = math.ceil(len(forget_encoded) / batch_size)
            steps_per_epoch = math.ceil(batches_per_epoch / accumulation_steps)
            total_steps = steps_per_epoch * epochs
            warmup_steps = steps_per_epoch
        else:
            total_steps, warmup_steps = 1, 0
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        rng = random.Random(args.seed)
        global_step = 0
        # PEFT initializes LoRA-A with zero gradient on the first step and
        # LoRA-B at zero; inspect every adapter tensor so the smoke checks an
        # actual update rather than whichever tensor happens to be first.
        smoke_initial = [p.detach().float().cpu().clone() for p in trainable] if args.smoke else None
        optimizer.zero_grad(set_to_none=True)
        with trace_path.open("w", encoding="utf-8") as trace:
            for epoch in range(epochs):
                order = list(range(len(forget_encoded)))
                rng.shuffle(order)
                micro_in_update = 0
                pending_losses = []
                for start in range(0, len(order), batch_size):
                    indices = order[start:start + batch_size]
                    if not indices:
                        continue
                    f_rows = [forget_encoded[i] for i in indices]
                    retain_indices = [rng.randrange(len(retain_encoded)) for _ in indices]
                    r_rows = [retain_encoded[i] for i in retain_indices]
                    f_ids, f_mask, f_labels = collate(f_rows, int(tokenizer.pad_token_id), input_device)
                    r_ids, r_mask, r_labels = collate(r_rows, int(tokenizer.pad_token_id), input_device)
                    f_logits = model(input_ids=f_ids, attention_mask=f_mask, use_cache=False).logits
                    f_current, _ = sequence_nll(f_logits, f_labels)
                    refs = torch.tensor([ref_nll[i] for i in indices], dtype=f_current.dtype, device=f_current.device)
                    forget_loss = objective_from_nlls(f_current, refs, METHOD["beta"])
                    if not torch.isfinite(forget_loss):
                        raise FloatingPointError("Non-finite NPO forget loss")
                    forget_loss_value = float(forget_loss.detach().cpu())
                    (METHOD["gamma_forget"] * forget_loss / accumulation_steps).backward()
                    del f_logits, f_current, refs, forget_loss

                    r_logits = model(input_ids=r_ids, attention_mask=r_mask, use_cache=False).logits
                    r_nll, r_counts = sequence_nll(r_logits, r_labels)
                    retain_loss = (r_nll.sum() / r_counts.sum().to(r_nll.dtype))
                    if not torch.isfinite(retain_loss):
                        raise FloatingPointError("Non-finite retain NLL")
                    retain_loss_value = float(retain_loss.detach().cpu())
                    (METHOD["alpha_retain"] * retain_loss / accumulation_steps).backward()
                    del r_logits, r_nll, r_counts, retain_loss
                    combined_loss_value = (
                        METHOD["gamma_forget"] * forget_loss_value
                        + METHOD["alpha_retain"] * retain_loss_value
                    )
                    if not math.isfinite(combined_loss_value):
                        raise FloatingPointError("Non-finite combined NPO+retain loss")
                    micro_in_update += 1
                    pending_losses.append({
                        "epoch": epoch + 1,
                        "forget_npo_loss": forget_loss_value,
                        "retain_token_nll": retain_loss_value,
                        "combined_loss": combined_loss_value,
                    })

                    final_micro = start + batch_size >= len(order)
                    if micro_in_update == accumulation_steps or final_micro:
                        # Correct for a short final accumulation window.
                        if final_micro and micro_in_update < accumulation_steps:
                            scale = accumulation_steps / micro_in_update
                            for parameter in trainable:
                                if parameter.grad is not None:
                                    parameter.grad.mul_(scale)
                        grad_norm = torch.nn.utils.clip_grad_norm_(trainable, METHOD["gradient_clip_norm"])
                        if not torch.isfinite(grad_norm):
                            raise FloatingPointError("Non-finite LoRA gradient norm")
                        optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)
                        global_step += 1
                        for record in pending_losses:
                            record.update({
                                "global_step": global_step,
                                "learning_rate": scheduler.get_last_lr()[0],
                                "gradient_norm": float(grad_norm.detach().cpu()),
                            })
                            trace.write(json.dumps(record) + "\n")
                        trace.flush()
                        pending_losses.clear()
                        micro_in_update = 0
                        if steps_to_run is not None and global_step >= steps_to_run:
                            break
                if steps_to_run is not None and global_step >= steps_to_run:
                    break

        if global_step == 0:
            raise RuntimeError("No optimizer update was performed")
        smoke_parameter_delta = None
        if args.smoke:
            smoke_parameter_delta = max(
                float((parameter.detach().float().cpu() - initial).abs().max())
                for parameter, initial in zip(trainable, smoke_initial)
            )
            if not math.isfinite(smoke_parameter_delta) or smoke_parameter_delta <= 0:
                raise RuntimeError("Real-data smoke loss was finite but no LoRA parameter changed")
        receipt.update({
            "status": "SMOKE_PASS" if args.smoke else "TRAINING_COMPLETE",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "global_steps": global_step,
            "trainable_parameters": sum(p.numel() for p in trainable),
            "all_parameters": sum(p.numel() for p in model.parameters()),
            "trainable_parameter_fraction": sum(p.numel() for p in trainable) / sum(p.numel() for p in model.parameters()),
            "optimizer_steps_with_finite_loss_and_gradient": global_step,
            "smoke_max_abs_lora_parameter_update": smoke_parameter_delta,
            "runtime_seconds": round(time.monotonic() - started, 2),
        })
        if not args.smoke:
            model.save_pretrained(output_dir / "adapter", safe_serialization=True)
            tokenizer.save_pretrained(output_dir / "adapter")
            receipt["adapter_saved"] = True
            receipt["adapter_files_sha256"] = {
                p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                for p in sorted((output_dir / "adapter").iterdir()) if p.is_file()
            }
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        print(json.dumps(receipt, indent=2), flush=True)
        return receipt
    except Exception as error:
        receipt.update({
            "status": "ERROR",
            "error": f"{type(error).__name__}: {error}",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "runtime_seconds": round(time.monotonic() - started, 2),
        })
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        raise


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", choices=MODEL_FAMILIES, required=True)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--objective-check", type=Path, required=True)
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    train(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/evaluate_npo_adapter.py
#!/usr/bin/env python3
"""Run the pinned official TOFU evaluator on one trained NPO LoRA adapter."""

from __future__ import annotations

import argparse
import hashlib
import json
import random
from collections import defaultdict
from pathlib import Path

from tofu_official_eval import (
    BOOTSTRAP_DRAWS,
    BOOTSTRAP_SEED,
    add_forget_quality,
    build_eval_sets,
    evaluate_checkpoint,
    load_data,
    summarize_examples,
)


PRIMARY_FIELD = "truth_ratio_score"
UTILITY_MARGINS = {
    "retain90.answer_probability": 0.05,
    "retain90.rougeL_recall": 0.05,
    "holdout10.answer_probability": 0.05,
}


def paired_cluster_delta(rows_a: list[dict], rows_b: list[dict], field: str, seed: int = BOOTSTRAP_SEED) -> dict:
    b_by_key = {(r["split"], r["source_row_index"]): r for r in rows_b}
    grouped = defaultdict(list)
    for row in rows_a:
        other = b_by_key.get((row["split"], row["source_row_index"]))
        if other is None or row.get(field) is None or other.get(field) is None:
            continue
        delta = float(other[field]) - float(row[field])
        grouped[row["profile_block"]].append(delta)
    if not grouped:
        raise ValueError(f"No matched cluster rows for {field}")
    cluster_ids = sorted(grouped)
    point = sum(v for values in grouped.values() for v in values) / sum(map(len, grouped.values()))
    rng = random.Random(seed)
    draws = []
    for _ in range(BOOTSTRAP_DRAWS):
        chosen = [rng.choice(cluster_ids) for _ in cluster_ids]
        sample = [v for group in chosen for v in grouped[group]]
        draws.append(sum(sample) / len(sample))
    draws.sort()
    return {
        "n_questions": sum(map(len, grouped.values())),
        "n_profile_blocks": len(grouped),
        "mean_delta_b_minus_a": point,
        "profile_cluster_bootstrap_95pct_ci": [
            draws[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
            draws[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
        ],
        "bootstrap_draws": BOOTSTRAP_DRAWS,
        "bootstrap_seed": seed,
    }


def compare_results(base: dict, adapter: dict, oracle: dict) -> dict:
    deltas = {}
    for split, metrics in {
        "forget10": ("answer_probability", "truth_ratio_score", "rougeL_recall"),
        "retain90": ("answer_probability", "truth_ratio_score", "rougeL_recall"),
        "holdout10": ("answer_probability", "truth_ratio_score", "rougeL_recall"),
    }.items():
        deltas[split] = {}
        for metric in metrics:
            if all(metric in row for row in base["per_example"][split]) and all(metric in row for row in adapter["per_example"][split]):
                deltas[split][f"npo_minus_full.{metric}"] = paired_cluster_delta(
                    base["per_example"][split], adapter["per_example"][split], metric
                )
            if split == "forget10" and all(metric in row for row in oracle["per_example"][split]):
                deltas[split][f"npo_minus_oracle.{metric}"] = paired_cluster_delta(
                    oracle["per_example"][split], adapter["per_example"][split], metric
                )

    utility_base = float(base["model_utility"]["harmonic_mean"])
    utility_adapter = float(adapter["model_utility"]["harmonic_mean"])
    primary = deltas["forget10"][f"npo_minus_full.{PRIMARY_FIELD}"]
    primary_pass = primary["profile_cluster_bootstrap_95pct_ci"][0] > 0.0
    utility_checks = {}
    for path, margin in UTILITY_MARGINS.items():
        split, field = path.split(".")
        delta = deltas[split][f"npo_minus_full.{field}"]
        low, high = delta["profile_cluster_bootstrap_95pct_ci"]
        utility_checks[path] = {
            "delta": delta["mean_delta_b_minus_a"],
            "ci95": [low, high],
            "noninferiority_margin": -margin,
            "pass": low > -margin,
        }
    utility_checks["model_utility.harmonic_mean"] = {
        "baseline": utility_base,
        "npo": utility_adapter,
        "delta": utility_adapter - utility_base,
        "noninferiority_margin": -0.05,
        "pass": utility_adapter - utility_base > -0.05,
        "interval": None,
        "note": "Official composite includes utility rows without profile IDs; point-difference gate only.",
    }
    return {
        "status": "PASS_DIAGNOSTIC_GATES" if primary_pass and all(x["pass"] for x in utility_checks.values()) else "FAIL_DIAGNOSTIC_GATES",
        "interpretation": "One-seed LoRA-NPO diagnostic only; not a full-parameter NPO reproduction or a confirmatory result.",
        "primary_outcome": "Forget10 Truth Ratio score (higher means perturbed and paraphrased answer likelihoods are less distinguishable; oracle proximity is assessed separately).",
        "primary_npo_minus_full": primary,
        "forget10_npo_minus_oracle": {
            metric: value for metric, value in deltas["forget10"].items() if "npo_minus_oracle" in metric
        },
        "cluster_bootstrap_deltas": deltas,
        "utility_noninferiority_gates": utility_checks,
        "primary_forget_gate_pass": primary_pass,
        "all_utility_gates_pass": all(x["pass"] for x in utility_checks.values()),
        "npo_model_utility_harmonic_mean": utility_adapter,
        "baseline_model_utility_harmonic_mean": utility_base,
        "forget10_summary": summarize_examples(adapter["per_example"]["forget10"]),
        "retain90_summary": summarize_examples(adapter["per_example"]["retain90"]),
        "holdout10_summary": summarize_examples(adapter["per_example"]["holdout10"]),
        "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": BOOTSTRAP_SEED, "unit": "profile block, paired by source row"},
    }


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", choices=("3b", "8b"), required=True)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--adapter-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    family_key = "llama_3_2_3b" if args.model == "3b" else "llama_3_1_8b"
    base_key = f"{family_key}_full"
    oracle_key = f"{family_key}_retain90"
    result_key = f"{family_key}_npo_lora_seed{args.seed}"
    base_path = args.output_dir / f"{base_key}_official_eval.json"
    oracle_path = args.output_dir / f"{oracle_key}_official_eval.json"
    if not base_path.is_file() or not oracle_path.is_file():
        raise FileNotFoundError("Verified full and retain90-oracle receipts are required before evaluating NPO")
    base = json.loads(base_path.read_text(encoding="utf-8"))
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    if base.get("status") != "PASS_OFFICIAL_TOFU_BASELINE" or oracle.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
        raise RuntimeError("Baseline receipts did not pass")
    data, hashes = load_data()
    if base.get("evaluation", {}).get("dataset_sha256_by_file") != hashes:
        raise RuntimeError("Pinned evaluation data hashes differ from the independently verified full-checkpoint baseline")
    training_receipt_path = args.adapter_dir.parent / "training_receipt.json"
    if not training_receipt_path.is_file():
        raise FileNotFoundError("NPO adapter directory must have its training_receipt.json in the parent directory")
    training_receipt = json.loads(training_receipt_path.read_text(encoding="utf-8"))
    if training_receipt.get("status") != "TRAINING_COMPLETE" or not training_receipt.get("adapter_saved"):
        raise RuntimeError("NPO adapter training receipt is not complete")
    adapter_files_sha256 = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(args.adapter_dir.iterdir()) if path.is_file()
    }
    if training_receipt.get("adapter_files_sha256") != adapter_files_sha256:
        raise RuntimeError("NPO adapter files do not match their training receipt")
    eval_sets = build_eval_sets(data)
    result = evaluate_checkpoint(
        base_key,
        args.output_dir,
        eval_sets,
        hashes,
        score_batch_size=1,
        generation_batch_size=1,
        max_memory_per_gpu_gib=12.5,
        adapter_path=args.adapter_dir,
        result_key=result_key,
        adapter_seed=args.seed,
    )
    add_forget_quality(args.output_dir, result_key, oracle_key)
    baseline = json.loads(base_path.read_text(encoding="utf-8"))
    adapter_eval = json.loads((args.output_dir / f"{result_key}_official_eval.json").read_text(encoding="utf-8"))
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    comparison = compare_results(baseline, adapter_eval, oracle)
    comparison.update({
        "model_family": family_key,
        "seed": args.seed,
        "base_checkpoint": base["model"],
        "dataset_revision": base["evaluation"]["dataset_revision"],
        "training_receipt": "training_receipt.json",
        "training_receipt_sha256": hashlib.sha256(training_receipt_path.read_bytes()).hexdigest(),
        "adapter_eval_receipt": f"{result_key}_official_eval.json",
    })
    (args.output_dir / "npo_comparison.json").write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
    print(json.dumps({"eval_status": result["status"], "comparison_status": comparison["status"], "comparison": comparison}, indent=2), flush=True)
    if result["status"] != "PASS_OFFICIAL_TOFU_ADAPTER_EVAL":
        raise RuntimeError("Adapter evaluation did not pass")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/verify_npo_run.py
#!/usr/bin/env python3
"""Independent integrity and metric checks for the one-seed TOFU NPO-LoRA pilot."""

from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
from collections import defaultdict
from pathlib import Path


DRAWS = 5000
SEED = 42
EPS = 1e-10
SPLIT_COUNTS = {"forget10": 400, "retain90": 400, "holdout10": 400, "real_authors": 100, "world_facts": 117}
MODEL_KEY = "llama_3_2_3b_full"
ORACLE_KEY = "llama_3_2_3b_retain90"
ADAPTER_KEY = "llama_3_2_3b_npo_lora_seed42"


def fail_if(condition: bool, message: str, problems: list[str]) -> None:
    if condition:
        problems.append(message)


def close(left: float, right: float, tolerance: float = 1e-9) -> bool:
    return math.isfinite(left) and math.isfinite(right) and abs(left - right) <= tolerance


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def paired_cluster_delta(rows_a: list[dict], rows_b: list[dict], field: str) -> dict:
    b_by_key = {(r["split"], r["source_row_index"]): r for r in rows_b}
    grouped: dict[int, list[float]] = defaultdict(list)
    for row in rows_a:
        other = b_by_key.get((row["split"], row["source_row_index"]))
        if other is None or field not in row or field not in other:
            continue
        grouped[row["profile_block"]].append(float(other[field]) - float(row[field]))
    if not grouped:
        raise ValueError(f"No matched profile blocks for {field}")
    blocks = sorted(grouped)
    values = [value for group in grouped.values() for value in group]
    point = sum(values) / len(values)
    rng = random.Random(SEED)
    draws = []
    for _ in range(DRAWS):
        sample_groups = [rng.choice(blocks) for _ in blocks]
        sample = [value for group in sample_groups for value in grouped[group]]
        draws.append(sum(sample) / len(sample))
    draws.sort()
    return {
        "n_questions": len(values),
        "n_profile_blocks": len(grouped),
        "mean_delta_b_minus_a": point,
        "profile_cluster_bootstrap_95pct_ci": [draws[int(.025 * (DRAWS - 1))], draws[int(.975 * (DRAWS - 1))]],
        "bootstrap_draws": DRAWS,
        "bootstrap_seed": SEED,
    }


def validate_metrics(result: dict, baseline: dict, protocol: dict, problems: list[str]) -> None:
    per_example = result.get("per_example", {})
    base_examples = baseline.get("per_example", {})
    for split, expected_count in SPLIT_COUNTS.items():
        rows = per_example.get(split, [])
        base_rows = base_examples.get(split, [])
        fail_if(len(rows) != expected_count, f"{split}: expected {expected_count} adapter rows, found {len(rows)}", problems)
        fail_if(len(base_rows) != expected_count, f"{split}: expected {expected_count} baseline rows, found {len(base_rows)}", problems)
        if len(rows) != len(base_rows):
            continue
        for index, (row, base_row) in enumerate(zip(rows, base_rows)):
            for field in ("source_row_index", "profile_block", "question_sha256", "source_answer"):
                fail_if(row.get(field) != base_row.get(field), f"{split}[{index}]: baseline alignment mismatch in {field}", problems)
            for field in ("answer_probability", "answer_token_loss", "answer_scored_tokens"):
                fail_if(field not in row, f"{split}[{index}]: missing {field}", problems)
            if not all(field in row for field in ("answer_probability", "answer_token_loss", "answer_scored_tokens")):
                continue
            probability = float(row["answer_probability"])
            token_loss = float(row["answer_token_loss"])
            fail_if(not (0 < probability <= 1) or not math.isfinite(token_loss) or token_loss < 0,
                    f"{split}[{index}]: invalid answer probability/loss", problems)
            fail_if(int(row["answer_scored_tokens"]) <= 0 or not close(probability, math.exp(-token_loss), 1e-7),
                    f"{split}[{index}]: answer probability does not match token loss", problems)
            if "rougeL_recall" in row:
                score = float(row["rougeL_recall"])
                fail_if(not math.isfinite(score) or score < 0 or score > 1,
                        f"{split}[{index}]: ROUGE-L outside [0,1]", problems)
            if "paraphrased_answer_token_loss" in row:
                para_loss = float(row["paraphrased_answer_token_loss"])
                para_probability = float(row.get("paraphrased_answer_probability", float("nan")))
                fail_if(not math.isfinite(para_loss) or para_loss < 0 or not close(para_probability, math.exp(-para_loss), 1e-7),
                        f"{split}[{index}]: paraphrased probability does not match token loss", problems)

            perturbed_losses = row.get("perturbed_answer_token_losses")
            if perturbed_losses is None:
                continue
            perturbed_losses = [float(x) for x in perturbed_losses]
            perturbed_probabilities = [float(x) for x in row.get("perturbed_answer_probabilities", [])]
            fail_if(not perturbed_losses or len(perturbed_losses) != len(perturbed_probabilities),
                    f"{split}[{index}]: perturbed answer loss/probability count mismatch", problems)
            if not perturbed_losses or len(perturbed_losses) != len(perturbed_probabilities):
                continue
            fail_if(any(not math.isfinite(x) or x < 0 for x in perturbed_losses),
                    f"{split}[{index}]: invalid perturbed token loss", problems)
            fail_if(any(not close(p, math.exp(-loss), 1e-7) for p, loss in zip(perturbed_probabilities, perturbed_losses)),
                    f"{split}[{index}]: perturbed probability does not match token loss", problems)
            denominator_field = "paraphrased_answer_probability" if "paraphrased_answer_probability" in row else "answer_probability"
            denominator = float(row[denominator_field])
            ratio = math.exp(-sum(perturbed_losses) / len(perturbed_losses)) / (denominator + EPS)
            expected_score = min(ratio, 1.0 / (ratio + EPS)) if split == "forget10" and denominator_field == "paraphrased_answer_probability" else max(0.0, 1.0 - ratio)
            fail_if(not close(float(row.get("truth_ratio", float("nan"))), ratio, 1e-7),
                    f"{split}[{index}]: Truth Ratio does not match saved token losses", problems)
            fail_if(not close(float(row.get("truth_ratio_score", float("nan"))), expected_score, 1e-7),
                    f"{split}[{index}]: Truth Ratio score does not match formula", problems)
            if "normalized_option_probability" in row:
                expected_option = probability / (probability + sum(perturbed_probabilities) + EPS)
                fail_if(not close(float(row["normalized_option_probability"]), expected_option, 1e-7),
                        f"{split}[{index}]: normalized option probability mismatch", problems)

    ev = result.get("evaluation", {})
    expected_hashes = protocol["dataset"]["evaluation_file_sha256"]
    fail_if(ev.get("dataset_sha256_by_file") != expected_hashes, "evaluation file hashes differ from locked protocol", problems)
    fail_if(ev.get("dataset_revision") != protocol["dataset"]["revision"], "evaluation dataset revision differs from locked protocol", problems)


def recompute_comparison(base: dict, adapter: dict, oracle: dict) -> dict:
    deltas = {}
    for split in ("forget10", "retain90", "holdout10"):
        deltas[split] = {}
        metrics = ("answer_probability", "truth_ratio_score", "rougeL_recall")
        for metric in metrics:
            if all(metric in row for row in base["per_example"][split]) and all(metric in row for row in adapter["per_example"][split]):
                deltas[split][f"npo_minus_full.{metric}"] = paired_cluster_delta(base["per_example"][split], adapter["per_example"][split], metric)
            if split == "forget10" and all(metric in row for row in oracle["per_example"][split]):
                deltas[split][f"npo_minus_oracle.{metric}"] = paired_cluster_delta(oracle["per_example"][split], adapter["per_example"][split], metric)
    primary = deltas["forget10"]["npo_minus_full.truth_ratio_score"]
    primary_pass = primary["profile_cluster_bootstrap_95pct_ci"][0] > 0
    gates = {}
    for path in ("retain90.answer_probability", "retain90.rougeL_recall", "holdout10.answer_probability"):
        split, field = path.split(".")
        delta = deltas[split][f"npo_minus_full.{field}"]
        low = delta["profile_cluster_bootstrap_95pct_ci"][0]
        gates[path] = {
            "delta": delta["mean_delta_b_minus_a"],
            "ci95": delta["profile_cluster_bootstrap_95pct_ci"],
            "noninferiority_margin": -0.05,
            "pass": low > -0.05,
        }
    utility_delta = float(adapter["model_utility"]["harmonic_mean"]) - float(base["model_utility"]["harmonic_mean"])
    gates["model_utility.harmonic_mean"] = {
        "baseline": float(base["model_utility"]["harmonic_mean"]),
        "npo": float(adapter["model_utility"]["harmonic_mean"]),
        "delta": utility_delta,
        "noninferiority_margin": -0.05,
        "pass": utility_delta > -0.05,
        "interval": None,
        "note": "Official composite includes utility rows without profile IDs; point-difference gate only.",
    }
    return {
        "primary_npo_minus_full": primary,
        "cluster_bootstrap_deltas": deltas,
        "utility_noninferiority_gates": gates,
        "primary_forget_gate_pass": primary_pass,
        "all_utility_gates_pass": all(item["pass"] for item in gates.values()),
        "status": "PASS_DIAGNOSTIC_GATES" if primary_pass and all(item["pass"] for item in gates.values()) else "FAIL_DIAGNOSTIC_GATES",
    }


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--training-dir", type=Path, required=True)
    parser.add_argument("--eval-dir", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--objective-check", type=Path, required=True)
    args = parser.parse_args()

    problems: list[str] = []
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    objective = json.loads(args.objective_check.read_text(encoding="utf-8"))
    receipt_path = args.training_dir / "training_receipt.json"
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    expected_protocol_hash = sha256(args.protocol)
    fail_if(protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN", "NPO protocol not preregistered", problems)
    fail_if(objective.get("status") != "PASS", "pinned objective value/gradient check failed", problems)
    fail_if(objective.get("protocol_sha256") != expected_protocol_hash, "objective check used a different protocol", problems)
    fail_if(receipt.get("status") != "TRAINING_COMPLETE", "full training receipt is not complete", problems)
    fail_if(receipt.get("run_type") != "exploratory_single_seed", "run is not the locked one-seed pilot", problems)
    fail_if(receipt.get("protocol_sha256") != expected_protocol_hash, "training protocol hash mismatch", problems)
    fail_if(receipt.get("objective_equivalence") != objective, "training receipt objective check differs from supplied check", problems)
    trainer_source = args.training_dir.parent / "npo_lora_train.py"
    fail_if(not trainer_source.is_file(), "trainer source snapshot missing beside training receipt", problems)
    if trainer_source.is_file():
        fail_if(receipt.get("training_script_sha256") != sha256(trainer_source), "trainer source hash mismatch", problems)

    model = receipt.get("model", {})
    expected_model = protocol["base_checkpoint"]
    fail_if(model.get("repo_id") != expected_model["repo_id"] or model.get("revision") != expected_model["revision"],
            "training checkpoint/revision mismatch", problems)
    fail_if(receipt.get("dataset", {}).get("repo") != protocol["dataset"]["repo_id"]
            or receipt.get("dataset", {}).get("revision") != protocol["dataset"]["revision"],
            "training dataset ID/revision mismatch", problems)
    fail_if(receipt.get("method", {}).get("beta") != protocol["objective"]["beta"]
            or receipt.get("method", {}).get("epochs") != protocol["adaptation"]["epochs"]
            or receipt.get("method", {}).get("lora_rank") != protocol["adaptation"]["lora_rank"],
            "training objective/hyperparameters differ from the locked protocol", problems)
    fail_if(receipt.get("seed") != protocol["seed"], "training seed mismatch", problems)
    expected_packages = {"transformers": "5.22.1", "accelerate": "1.15.0", "peft": "0.21.2", "scipy": "1.16.3"}
    for package, version in expected_packages.items():
        fail_if(receipt.get("package_versions", {}).get(package) != version,
                f"training package version mismatch for {package}", problems)
    fail_if(receipt.get("package_versions", {}).get("torchao") is not None,
            "optional torchao must be absent for this unquantized LoRA run", problems)
    fail_if(not str(receipt.get("package_versions", {}).get("torch", "")), "missing torch version", problems)
    fail_if(receipt.get("global_steps") != 500, f"expected 500 completed optimizer steps, found {receipt.get('global_steps')}", problems)
    fail_if(not receipt.get("training_started") or not receipt.get("adapter_saved"), "training or adapter save receipt missing", problems)
    fail_if(receipt.get("runtime", {}).get("cpu_or_disk_offload") is not False, "training used or failed to rule out CPU/disk offload", problems)
    fail_if(len(receipt.get("gpu_names", [])) < 2 or receipt.get("gpu_names", []).count("Tesla T4") < 2,
            "training receipt does not show Kaggle T4 x2", problems)
    fail_if(receipt.get("tokenization", {}).get("truncated_examples") != 0, "training data was truncated", problems)
    fail_if(receipt.get("tokenization", {}).get("max_length_observed", 999999) > 512, "training sequence exceeded locked maximum length", problems)
    train_data = receipt.get("dataset", {})
    locked_training_files = protocol["dataset"]["training"]
    fail_if(train_data.get("forget10_sha256") != locked_training_files["forget_file"]["sha256"], "forget10 training data hash mismatch", problems)
    fail_if(train_data.get("retain90_sha256") != locked_training_files["retain_file"]["sha256"], "retain90 training data hash mismatch", problems)
    fail_if(train_data.get("forget10_rows") != 400 or train_data.get("retain90_rows") != 3600,
            "training data row counts differ from protocol", problems)
    device_map = receipt.get("runtime", {}).get("device_map", {})
    fail_if(not device_map or any(str(device).lower() in {"cpu", "disk"} for device in device_map.values()),
            "training device map is missing or contains CPU/disk placement", problems)

    adapter_dir = args.training_dir / "adapter"
    recorded_adapter_hashes = receipt.get("adapter_files_sha256", {})
    actual_adapter_hashes = {path.name: sha256(path) for path in sorted(adapter_dir.iterdir()) if path.is_file()} if adapter_dir.is_dir() else {}
    fail_if(not actual_adapter_hashes or actual_adapter_hashes != recorded_adapter_hashes, "adapter file hashes do not match training receipt", problems)
    trace_path = args.training_dir / "loss_trace.jsonl"
    trace_rows = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    fail_if(len(trace_rows) != 4000, f"expected 4000 per-forget loss trace rows, found {len(trace_rows)}", problems)
    for index, row in enumerate(trace_rows):
        for name in ("forget_npo_loss", "retain_token_nll", "learning_rate", "gradient_norm"):
            fail_if(name not in row or not math.isfinite(float(row[name])), f"loss trace row {index} has invalid {name}", problems)

    base_path = args.eval_dir / f"{MODEL_KEY}_official_eval.json"
    oracle_path = args.eval_dir / f"{ORACLE_KEY}_official_eval.json"
    adapter_path = args.eval_dir / f"{ADAPTER_KEY}_official_eval.json"
    comparison_path = args.eval_dir / "npo_comparison.json"
    base, oracle, adapter, comparison = [json.loads(path.read_text(encoding="utf-8")) for path in (base_path, oracle_path, adapter_path, comparison_path)]
    fail_if(base.get("status") != "PASS_OFFICIAL_TOFU_BASELINE" or oracle.get("status") != "PASS_OFFICIAL_TOFU_BASELINE",
            "3B full/oracle baseline receipts do not pass", problems)
    fail_if(adapter.get("status") != "PASS_OFFICIAL_TOFU_ADAPTER_EVAL", "NPO adapter evaluation did not pass", problems)
    fail_if(adapter.get("model", {}).get("adapter", {}).get("files_sha256") != actual_adapter_hashes,
            "evaluated adapter hashes differ from trained adapter", problems)
    fail_if(adapter.get("model", {}).get("repo_id") != expected_model["repo_id"] or adapter.get("model", {}).get("revision") != expected_model["revision"],
            "adapter evaluation checkpoint/revision mismatch", problems)
    fail_if(adapter.get("model", {}).get("adapter", {}).get("seed") != protocol["seed"],
            "adapter evaluation seed differs from locked protocol", problems)
    fail_if(adapter.get("runtime", {}).get("cpu_offload_used") is not False or adapter.get("runtime", {}).get("research_adapter_created") is not True,
            "adapter evaluation placement or adapter receipt invalid", problems)
    validate_metrics(adapter, base, protocol, problems)

    recomputed = recompute_comparison(base, adapter, oracle)
    for name in ("primary_npo_minus_full", "cluster_bootstrap_deltas", "utility_noninferiority_gates"):
        if comparison.get(name) != recomputed[name]:
            problems.append(f"recorded comparison does not reproduce for {name}")
    if comparison.get("status") != recomputed["status"]:
        problems.append("recorded diagnostic-gate status does not reproduce")
    fail_if(comparison.get("training_receipt_sha256") != sha256(receipt_path), "comparison refers to a different training receipt", problems)

    verification = {
        "overall_status": "PASS" if not problems else "FAIL",
        "research_diagnostic_gate_status": recomputed["status"],
        "research_interpretation": "Single-seed exploratory NPO-LoRA diagnostic. Artifact integrity may pass even when the prespecified scientific/utility gates fail.",
        "problems": problems,
        "training_receipt_sha256": sha256(receipt_path),
        "protocol_sha256": expected_protocol_hash,
        "objective_equivalence": objective,
        "verified_splits": SPLIT_COUNTS,
        "primary_result": recomputed["primary_npo_minus_full"],
        "utility_gates": recomputed["utility_noninferiority_gates"],
    }
    out = args.eval_dir / "npo_verification.json"
    out.write_text(json.dumps(verification, indent=2) + "\n", encoding="utf-8")
    report = [
        "# TOFU NPO-LoRA pilot verification",
        "",
        f"Artifact verification: **{verification['overall_status']}**",
        f"Prespecified diagnostic gates: **{verification['research_diagnostic_gate_status']}**",
        "",
        "A PASS here means provenance, saved outputs, official per-row formulas, paired profile-bootstrap results, and artifact hashes reproduce. It does not by itself mean the NPO method passed its scientific or utility gates.",
        "",
        "## Primary outcome",
        "",
        f"NPO minus full Truth Ratio score: {recomputed['primary_npo_minus_full']['mean_delta_b_minus_a']:.6f} "
        f"[{recomputed['primary_npo_minus_full']['profile_cluster_bootstrap_95pct_ci'][0]:.6f}, "
        f"{recomputed['primary_npo_minus_full']['profile_cluster_bootstrap_95pct_ci'][1]:.6f}], "
        f"{recomputed['primary_npo_minus_full']['n_questions']} rows across "
        f"{recomputed['primary_npo_minus_full']['n_profile_blocks']} profile blocks.",
        "",
        "## Utility gates",
        "",
    ]
    for name, gate in recomputed["utility_noninferiority_gates"].items():
        report.append(f"- `{name}`: delta {gate['delta']:.6f}, pass={gate['pass']}.")
    if problems:
        report.extend(["", "## Verification problems", ""])
        report.extend(f"- {problem}" for problem in problems)
    (args.eval_dir / "npo_verification.md").write_text("\n".join(report) + "\n", encoding="utf-8")
    print(json.dumps(verification, indent=2))
    if problems:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/npo_lora_pilot_protocol.json
{
  "protocol_id": "tofu_3b_npo_lora_pilot_v1",
  "status": "PREREGISTERED_BEFORE_GPU_RUN",
  "registered_locally_at": "2026-11-07",
  "research_question": "On TOFU forget10, does a resource-bounded LoRA implementation of the pinned Open-Unlearning NPO objective move the 3B model's official Truth Ratio score toward the retain90-only oracle while preserving retain and holdout utility?",
  "study_type": "Exploratory one-seed implementation pilot; not confirmatory evidence and not a full-parameter reproduction.",
  "model_family": "3b",
  "seed": 42,
  "base_checkpoint": {
    "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    "architecture_id": "meta-llama/Llama-3.2-3B-Instruct"
  },
  "retain_only_reference_checkpoint": {
    "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
    "revision": "2c35819e9a731d096df9abd6efbcc361870c5333"
  },
  "dataset": {
    "repo_id": "locuslab/TOFU",
    "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
    "training": {
      "forget_file": {"name": "forget10.json", "rows": 400, "sha256": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6"},
      "retain_file": {"name": "retain90.json", "rows": 3600, "sha256": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55"},
      "holdout_used_for_training": false
    },
    "evaluation_rows": {"forget10": 400, "retain90": 400, "holdout10": 400, "real_authors": 100, "world_facts": 117},
    "evaluation_file_sha256": {
      "forget10.json": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6",
      "forget10_perturbed.json": "6fbcb946c57ea1d7b2124cea0e61bf3b5409d1bc10d02368f090109450ed73c7",
      "retain90.json": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55",
      "retain_perturbed.json": "fc69f33bad70d3dca65920bdb54380039e3e8872dd16bbae20c1a839b99533fb",
      "holdout10.json": "efec32c7a7f66bfefaa90b8a4cc583296b0bbf882495db61076dec2d6edf44dd",
      "real_authors_perturbed.json": "a23255383ac75a5fbe5acd57622a03fb9c368954fd7b2428afa152b784d9533a",
      "world_facts_perturbed.json": "0e171838040d0ec94b1ce248d891508e85e3b77344370077caa27b61898eb3e1"
    }
  },
  "objective": {
    "source": "locuslab/open-unlearning@2d98027677f3468d52b6baa3fc4f2b9f2dc08820",
    "method": "NPO forget objective plus retain negative log-likelihood",
    "formula": "mean[-2/beta * log_sigmoid(beta * (current_sequence_summed_nll - reference_sequence_summed_nll))] + retain_token_mean_nll",
    "beta": 0.1,
    "forget_weight_gamma": 1.0,
    "retain_weight_alpha": 1.0,
    "reference_model": "frozen initial full checkpoint; reference sequence NLL precomputed before LoRA insertion",
    "equivalence_requirement": "Pinned upstream loss value and parameter gradients match the candidate implementation within absolute tolerance 1e-6 on the deterministic CPU fixture before any GPU training."
  },
  "adaptation": {
    "scope": "LoRA adapters only; frozen FP16 base; report as an NPO-LoRA adaptation, not full-parameter NPO.",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "precision": "FP16 frozen base checkpoint; FP32 LoRA parameters and loss; SDPA",
    "optimizer": "AdamW, betas=(0.9,0.999), eps=1e-8, weight_decay=0",
    "learning_rate": 0.00001,
    "epochs": 10,
    "effective_paired_batch_size": 8,
    "forget_retain_pairing": "one uniformly sampled retain example with replacement for each forget example; seed 42",
    "scheduler": "linear; one epoch warmup",
    "warmup": "one epoch of optimizer steps, then linear decay",
    "gradient_clip_norm": 1.0,
    "max_length": 512,
    "truncation": false,
    "hardware": "Kaggle GPU T4 x2 only; refuse CPU/disk weight offload; FP16 base, FP32 adapter and loss; SDPA",
    "determinism_note": "Python, CPU Torch, and CUDA Torch RNGs are seeded; CUDA kernels may remain nondeterministic."
  },
  "bounded_smoke": {
    "required_before_full_pilot": true,
    "real_data_rows": "first eight pinned forget10 rows and one sampled retain row",
    "optimizer_steps": 1,
    "success_criteria": ["finite NPO/retain losses and gradients", "exactly one optimizer step", "nonzero LoRA parameter update", "T4 x2 placement", "no CPU/disk offload", "smoke adapter discarded and never scored as research result"]
  },
  "evaluation": {
    "official_metrics": ["answer_probability", "greedy-generation ROUGE-L recall", "Truth Ratio score", "normalized option probability", "nine-component model-utility harmonic mean"],
    "primary_metric": "forget10 mean Truth Ratio score, paired NPO adapter minus the same 3B full-checkpoint rows",
    "primary_direction": "positive values indicate movement toward the retain-only oracle: the score is higher when perturbed and paraphrased answer likelihoods are less distinguishable; verify against the measured full-to-oracle contrast.",
    "uncertainty": "paired percentile bootstrap resampling the 20 TOFU profile blocks, 5000 draws, seed 42; preserve all rows in each sampled block",
    "primary_signal_rule": "Exploratory signal only if the 95% profile-block bootstrap interval for NPO-minus-full Truth Ratio score is wholly above zero.",
    "utility_gates": {
      "retain90.answer_probability": {"minimum_delta": -0.05, "rule": "95% paired profile-block interval lower bound strictly greater than -0.05"},
      "retain90.rougeL_recall": {"minimum_delta": -0.05, "rule": "95% paired profile-block interval lower bound strictly greater than -0.05"},
      "holdout10.answer_probability": {"minimum_delta": -0.05, "rule": "95% paired profile-block interval lower bound strictly greater than -0.05"},
      "model_utility.harmonic_mean": {"minimum_delta": -0.05, "rule": "point difference strictly greater than -0.05; descriptive gate because utility rows do not share profile IDs"}
    },
    "secondary_descriptive_metrics": ["forget10 answer probability and ROUGE-L change", "NPO-versus-oracle paired contrasts", "all individual utility components", "profile-cluster intervals"]
  },
  "stopping_rules": [
    "Do not train unless the pinned upstream objective/gradient check passes and the 3B baseline full/oracle outputs independently verify.",
    "Do not train unless the bounded real-data smoke passes with a nonzero adapter update and GPU-only placement.",
    "Stop and preserve partial receipts on non-finite loss/gradient, OOM, missing data, revision/hash mismatch, truncation, offload, or incomplete optimizer steps.",
    "Do not tune after inspecting pilot evaluation results; any change requires a new protocol and new seed.",
    "A failed primary or utility gate is a diagnostic result, not a reason to change the threshold or discard the run."
  ],
  "interpretation_limits": [
    "Single seed and one model family: exploratory only, cannot establish general effectiveness.",
    "Adapter training changes capacity and optimization relative to the original full-parameter recipe.",
    "TOFU profiles are fictional and do not support claims about deletion of real personal data or real-world privacy.",
    "Truth Ratio is one benchmark metric; this run is not proof of complete knowledge removal."
  ],
  "code_sources": {
    "trainer": "npo_lora_train.py",
    "objective": "npo_objective.py",
    "objective_verifier": "verify_npo_objective.py",
    "official_evaluator": "tofu_official_eval.py",
    "adapter_evaluator": "evaluate_npo_adapter.py",
    "independent_run_verifier": "verify_npo_run.py"
  }
}


In [ ]:
%%writefile /kaggle/working/requirements-npo-pilot.txt
# Kaggle runtime packages used by the bounded TOFU NPO-LoRA pilot.
# Keep the preinstalled CUDA-enabled torch build supplied by Kaggle.
transformers==5.22.1
accelerate==1.15.0
peft==0.21.2
scipy==1.16.3
rouge-score==0.1.2


In [ ]:
subprocess.run([sys.executable, '/kaggle/working/verify_npo_objective.py', '--upstream-src', '/kaggle/working/open-unlearning/src', '--protocol', '/kaggle/working/npo_lora_pilot_protocol.json', '--output', '/kaggle/working/npo_objective_equivalence.json'], check=True)
objective_check = json.loads(Path('/kaggle/working/npo_objective_equivalence.json').read_text())
if objective_check.get('status') != 'PASS': raise RuntimeError('Pinned Open-Unlearning NPO loss/gradient equivalence did not pass')
print({'objective_check': objective_check})


In [ ]:
input_root = Path('/kaggle/input')
partial_manifests = list(input_root.rglob('partial_resume_manifest.json'))
upload_manifests = list(input_root.rglob('upload_manifest.json'))
if len(partial_manifests) != 1 or len(upload_manifests) != 1:
    raise RuntimeError(f'Attach exactly the verified resume dataset; partial={partial_manifests}, upload={upload_manifests}')
partial_path, upload_path = partial_manifests[0], upload_manifests[0]
def file_sha(path):
    import hashlib
    return hashlib.sha256(path.read_bytes()).hexdigest()
if file_sha(upload_path) != 'de117326805e5b1b77fe6f18de94f9dac5e8ee3f369ff056ed3a1f762bbfa0d2': raise RuntimeError('Pinned upload manifest checksum mismatch')
if file_sha(partial_path) != '4aa0a4d65c102feb703c057daf03023f2fec3facf7e6fe790c655b635c7648ff': raise RuntimeError('Pinned partial resume manifest checksum mismatch')
upload_receipt = json.loads(upload_path.read_text())
if upload_receipt.get('archive_sha256') != '5882fc3ebf8299413b60503edd9eeee080724909f071f5d3ba34dae683f8ea70' or upload_receipt.get('archive_crc') != 'PASS': raise RuntimeError('The uploaded baseline archive receipt is not the expected CRC-passing archive')
manifest = json.loads(partial_path.read_text())
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ROOT = Path('/kaggle/working') / ('tofu_npo_run_' + run_id)
BASELINE_DIR = RUN_ROOT / 'tofu_official_eval'
TRAIN_DIR = RUN_ROOT / 'npo_train'
SMOKE_DIR = RUN_ROOT / 'npo_smoke'
BASELINE_DIR.mkdir(parents=True, exist_ok=False)
copied = []
for relative, item in manifest['files'].items():
    source = partial_path.parent / relative
    if not source.is_file() or file_sha(source) != item['sha256']:
        raise RuntimeError(f'Restored baseline input missing or hash-invalid: {relative}')
    target = BASELINE_DIR / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    copied.append(relative)
shutil.copy2(partial_path, BASELINE_DIR / 'partial_resume_manifest.json')
for name in ['tofu_official_eval.py', 'verify_tofu_official_eval.py', 'npo_objective.py', 'verify_npo_objective.py', 'npo_lora_train.py', 'evaluate_npo_adapter.py', 'verify_npo_run.py', 'npo_lora_pilot_protocol.json', 'requirements-npo-pilot.txt']:
    shutil.copy2(Path('/kaggle/working') / name, RUN_ROOT / name)
audit_path = BASELINE_DIR / 'audit' / 'tofu_readiness_audit.json'
audit = json.loads(audit_path.read_text())
if audit.get('data_audit', {}).get('checks', {}).get('structural_checks_pass') is not True:
    raise RuntimeError('Pinned baseline data audit did not pass')
verify = subprocess.run([sys.executable, '/kaggle/working/verify_tofu_official_eval.py', str(BASELINE_DIR), '--audit', str(audit_path), '--allow-partial'], capture_output=True, text=True)
(RUN_ROOT / 'baseline_verification.stdout.json').write_text(verify.stdout + verify.stderr)
if verify.returncode: raise RuntimeError('Baseline partial verifier failed: ' + verify.stdout[-3000:])
baseline_verification = json.loads((BASELINE_DIR / 'official_eval_verification.json').read_text())
for key in ['llama_3_2_3b_full', 'llama_3_2_3b_retain90']:
    if baseline_verification['models'][key]['status'] != 'PASS': raise RuntimeError(f'3B prerequisite failed: {key}')
print({'run_root': str(RUN_ROOT), 'restored_baseline_files': copied, 'baseline_verification': baseline_verification['overall_status'], 'required_3b_baselines': 'PASS'})


In [ ]:
def archive_run():
    archive = Path('/kaggle/working/tofu_npo_lora_pilot_artifacts.zip')
    if archive.exists(): archive.unlink()
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for path in sorted(RUN_ROOT.rglob('*')):
            if path.is_file(): zf.write(path, path.relative_to(RUN_ROOT.parent))
    print({'artifact_zip': str(archive), 'bytes': archive.stat().st_size})

smoke_cmd = [sys.executable, '/kaggle/working/npo_lora_train.py', '--model', '3b', '--seed', '42', '--protocol', '/kaggle/working/npo_lora_pilot_protocol.json', '--objective-check', '/kaggle/working/npo_objective_equivalence.json', '--output-dir', str(SMOKE_DIR), '--smoke']
smoke = subprocess.run(smoke_cmd, capture_output=True, text=True)
(RUN_ROOT / 'smoke.stdout.log').write_text(smoke.stdout + smoke.stderr)
archive_run()
if smoke.returncode: raise RuntimeError('Bounded real-data smoke failed; receipt/log preserved in the artifact ZIP. ' + smoke.stderr[-4000:])
smoke_receipt = json.loads((SMOKE_DIR / 'training_receipt.json').read_text())
if smoke_receipt.get('status') != 'SMOKE_PASS' or smoke_receipt.get('global_steps') != 1 or smoke_receipt.get('smoke_max_abs_lora_parameter_update', 0) <= 0:
    raise RuntimeError('Smoke receipt did not pass all one-step/update checks')
if smoke_receipt.get('runtime', {}).get('cpu_or_disk_offload') is not False:
    raise RuntimeError('Smoke used CPU/disk offload')
print({'bounded_smoke': 'PASS', 'global_steps': smoke_receipt['global_steps'], 'max_abs_adapter_update': smoke_receipt['smoke_max_abs_lora_parameter_update'], 'research_scores_created': False, 'full_training_enabled': RUN_FULL_TRAINING})


In [ ]:
if RUN_FULL_TRAINING:
    train_cmd = [sys.executable, '/kaggle/working/npo_lora_train.py', '--model', '3b', '--seed', '42', '--protocol', '/kaggle/working/npo_lora_pilot_protocol.json', '--objective-check', '/kaggle/working/npo_objective_equivalence.json', '--output-dir', str(TRAIN_DIR)]
    train = subprocess.run(train_cmd, capture_output=True, text=True)
    (RUN_ROOT / 'training.stdout.log').write_text(train.stdout + train.stderr)
    archive_run()
    if train.returncode: raise RuntimeError('NPO-LoRA pilot training failed; partial receipt and trace were archived. ' + train.stderr[-5000:])
    train_receipt = json.loads((TRAIN_DIR / 'training_receipt.json').read_text())
    if train_receipt.get('status') != 'TRAINING_COMPLETE' or train_receipt.get('global_steps') != 500:
        raise RuntimeError(f'Full pilot did not meet its 500-step completion criteria: {train_receipt}')
    eval_cmd = [sys.executable, '/kaggle/working/evaluate_npo_adapter.py', '--model', '3b', '--seed', '42', '--adapter-dir', str(TRAIN_DIR / 'adapter'), '--output-dir', str(BASELINE_DIR)]
    evaluation = subprocess.run(eval_cmd, capture_output=True, text=True)
    (RUN_ROOT / 'evaluation.stdout.log').write_text(evaluation.stdout + evaluation.stderr)
    archive_run()
    if evaluation.returncode: raise RuntimeError('Official NPO evaluation failed; training and partial score receipts were archived. ' + evaluation.stderr[-5000:])
    verify_cmd = [sys.executable, '/kaggle/working/verify_npo_run.py', '--training-dir', str(TRAIN_DIR), '--eval-dir', str(BASELINE_DIR), '--protocol', '/kaggle/working/npo_lora_pilot_protocol.json', '--objective-check', '/kaggle/working/npo_objective_equivalence.json']
    verification = subprocess.run(verify_cmd, capture_output=True, text=True)
    (RUN_ROOT / 'npo_verification.stdout.log').write_text(verification.stdout + verification.stderr)
    archive_run()
    if verification.returncode: raise RuntimeError('Independent NPO artifact verification failed; see archived npo_verification.json. ' + verification.stderr[-5000:])
    print('NPO-LoRA pilot finished. Inspect npo_verification.md and npo_comparison.json; artifact verification and research diagnostic gates are reported separately.')
else:
    print('Smoke-only version complete. Review the archived one-step receipt before building the full-training version.')
